In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Figure-3-style proper-motion visualization for COSMOS2025 ID 639188.

This script is designed to make a Bradač et al. (2026) Fig. 3-like figure
for one brown-dwarf candidate and all available same-filter epoch pairs.

It uses:
  1) the image cutout FITS files already downloaded in your previous PM notebook;
  2) the image_level_pm_summary.csv already produced by your previous image-level code;
  3) the candidate coordinate.

Main output:
  - one PNG per epoch pair;
  - one multi-page PDF containing all pairs;
  - a compact PM summary CSV with approximate uncertainties.

Important:
  The uncertainty here is an image-registration diagnostic. For the final paper,
  use fake-source injection/recovery to estimate the candidate centroid uncertainty
  in each image, following the Bradač et al. Fig. 3 analysis logic.
"""

from __future__ import annotations

import copy
import math
import os
import re
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Circle

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.coordinates import SkyCoord
from astropy.nddata import Cutout2D
from astropy.stats import sigma_clip
from astropy.time import Time
import astropy.units as u
from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch, LinearStretch

try:
    from photutils.centroids import centroid_2dg, centroid_com
    HAVE_PHOTUTILS = True
except Exception:
    HAVE_PHOTUTILS = False

try:
    from scipy.ndimage import shift as ndi_shift
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

try:
    from reproject import reproject_interp
    HAVE_REPROJECT = True
except Exception:
    HAVE_REPROJECT = False


# =============================================================================
# USER CONFIG
# =============================================================================

OBJID = 639188
TARGET = SkyCoord(150.0634341, 2.3552216, unit="deg")

# Directory used in your previous notebook.
CUTOUT_DIR = Path("jwst_image_cutouts_for_pm")

# Your previous image-level output. The script tries both paths.
SUMMARY_CSV_CANDIDATES = [
    Path("proper_motion_image_level/image_level_pm_summary.csv"),
    Path("image_level_pm_summary.csv"),
    Path("/mnt/data/image_level_pm_summary.csv"),
]

OUTDIR = Path("ID639188_Fig3_style_PM")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Image definitions from your previous notebook.
IMAGE_EPOCHS: Dict[str, Dict[str, str]] = {
    "PRIMER_2023_F115W": {
        "cutout": str(CUTOUT_DIR / "PRIMER_2023_F115W_cutout.fits"),
        "date": "2023-04-27",
        "label": "PRIMER F115W 2023",
    },
    "COSMOSWeb_2023_F115W": {
        "cutout": str(CUTOUT_DIR / "COSMOSWeb_2023_F115W_cutout.fits"),
        "date": "2023-12-30",
        "label": "COSMOS-Web F115W 2023",
    },
    "COSMOS3D_2025_F115W": {
        "cutout": str(CUTOUT_DIR / "COSMOS3D_2025_F115W_cutout.fits"),
        "date": "2025-04-15",
        "label": "COSMOS-3D F115W 2025",
    },
    "PRIMER_2023_F200W": {
        "cutout": str(CUTOUT_DIR / "PRIMER_2023_F200W_cutout.fits"),
        "date": "2023-04-27",
        "label": "PRIMER F200W 2023",
    },
    "COSMOS3D_2025_F200W": {
        "cutout": str(CUTOUT_DIR / "COSMOS3D_2025_F200W_cutout.fits"),
        "date": "2025-04-15",
        "label": "COSMOS-3D F200W 2025",
    },
    "PRIMER_2023_F356W": {
        "cutout": str(CUTOUT_DIR / "PRIMER_2023_F356W_cutout.fits"),
        "date": "2023-04-27",
        "label": "PRIMER F356W 2023",
    },
    "COSMOS3D_2025_F356W": {
        "cutout": str(CUTOUT_DIR / "COSMOS3D_2025_F356W_cutout.fits"),
        "date": "2025-04-15",
        "label": "COSMOS-3D F356W 2025",
    },
}

# If None, all rows in image_level_pm_summary.csv are plotted.
# You can manually restrict pairs here if desired.
PAIRS_TO_PLOT: Optional[list[tuple[str, str]]] = None
# Example:
# PAIRS_TO_PLOT = [
#     ("PRIMER_2023_F115W", "COSMOS3D_2025_F115W"),
#     ("PRIMER_2023_F200W", "COSMOS3D_2025_F200W"),
#     ("PRIMER_2023_F356W", "COSMOS3D_2025_F356W"),
# ]

# Figure geometry, matching the spirit of Bradač et al. Fig. 3.
FULL_SIZE_ARCSEC = 16.0
ZOOM_SIZE_ARCSEC = 1.8
CENTROID_BOX_PIX = 13
APERTURE_RADIUS_ARCSEC = 0.22
TARGET_EXCLUSION_ARCSEC = 1.0

# Difference-image options.
# If True, image 2 is scaled to image 1 using a robust pixel slope before subtraction.
AUTO_FLUX_SCALE_FOR_DIFF = True

# For a final science figure, prefer the longest same-filter baselines.
MIN_BASELINE_FOR_MAIN_LABEL_YR = 1.0


# =============================================================================
# BASIC UTILITIES
# =============================================================================

def find_summary_csv() -> Path:
    for p in SUMMARY_CSV_CANDIDATES:
        if p.exists():
            return p
    raise FileNotFoundError(
        "Could not find image_level_pm_summary.csv. "
        "Run your previous image-level proper-motion notebook first, or edit SUMMARY_CSV_CANDIDATES."
    )


def filter_from_epoch_name(name: str) -> str:
    m = re.search(r"(F\d+[MW])", name.upper())
    return m.group(1) if m else "UNKNOWN"


def scalar_float(x) -> float:
    arr = np.asarray(x).ravel()
    if arr.size != 1:
        raise ValueError(f"Expected scalar, got shape {np.asarray(x).shape}")
    return float(arr[0])


def robust_median_sigma(x: np.ndarray) -> Tuple[float, float]:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return np.nan, np.nan
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    sig = 1.4826 * mad
    if not np.isfinite(sig) or sig <= 0:
        sig = np.nanstd(x)
    return float(med), float(sig)


def sigma_clipped_background(data: np.ndarray) -> float:
    arr = np.asarray(data, dtype=float)
    good = np.isfinite(arr)
    if np.sum(good) < 10:
        return 0.0
    clipped = sigma_clip(arr[good], sigma=3.0, maxiters=5)
    med = np.nanmedian(clipped.data[~clipped.mask]) if np.any(~clipped.mask) else np.nanmedian(arr[good])
    return float(med) if np.isfinite(med) else 0.0


def subtract_background(data: np.ndarray) -> np.ndarray:
    return np.asarray(data, dtype=float) - sigma_clipped_background(data)


def find_science_hdu(path: Path) -> Tuple[np.ndarray, fits.Header, int]:
    """Find the first 2D science-like image HDU."""
    with fits.open(path, memmap=False) as hdul:
        preferred = []
        for i, h in enumerate(hdul):
            name = str(getattr(h, "name", "")).upper()
            if h.data is not None and np.asarray(h.data).ndim == 2:
                if name in {"SCI", "PRIMARY"}:
                    preferred.append(i)
        candidates = preferred if preferred else [
            i for i, h in enumerate(hdul)
            if h.data is not None and np.asarray(h.data).ndim == 2
        ]
        if not candidates:
            raise ValueError(f"No 2D image HDU found in {path}")
        i = candidates[0]
        data = np.asarray(hdul[i].data, dtype=float)
        header = hdul[i].header.copy()
        # Sometimes WCS is in primary header while image is in extension.
        if i != 0:
            header = hdul[0].header.copy() + header
        return data, header, i


def load_image(epoch_name: str) -> Tuple[np.ndarray, WCS, fits.Header]:
    path = Path(IMAGE_EPOCHS[epoch_name]["cutout"])
    if not path.exists():
        raise FileNotFoundError(f"Missing cutout for {epoch_name}: {path}")
    data, header, _ = find_science_hdu(path)
    wcs = WCS(header)
    data = subtract_background(data)
    return data, wcs, header


def pixel_scale_arcsec(wcs: WCS) -> float:
    scales = proj_plane_pixel_scales(wcs) * 3600.0
    return float(np.nanmedian(np.abs(scales)))


def world_to_pixel(wcs: WCS, coord: SkyCoord) -> Tuple[float, float]:
    x, y = wcs.world_to_pixel(coord)
    return scalar_float(x), scalar_float(y)


def skycoord_with_offsets(base: SkyCoord, dra_mas: float, ddec_mas: float) -> SkyCoord:
    """
    Offset convention:
    dra_mas is ΔRA*cos(Dec) in mas, positive east.
    ddec_mas is ΔDec in mas, positive north.
    """
    return base.spherical_offsets_by(dra_mas * u.mas, ddec_mas * u.mas)


def crop_around_xy(data: np.ndarray, x: float, y: float, half_size: int) -> Tuple[np.ndarray, int, int]:
    ny, nx = data.shape
    xi, yi = int(round(x)), int(round(y))
    x0 = max(0, xi - half_size)
    x1 = min(nx, xi + half_size + 1)
    y0 = max(0, yi - half_size)
    y1 = min(ny, yi + half_size + 1)
    return data[y0:y1, x0:x1], x0, y0


def centroid_source(data: np.ndarray, wcs: WCS, coord_guess: SkyCoord,
                    box_size: int = CENTROID_BOX_PIX) -> Tuple[SkyCoord, float, float, str]:
    """
    Measure centroid near coord_guess. Returns centroid SkyCoord, x, y in the input image,
    and a status string.
    """
    x0, y0 = world_to_pixel(wcs, coord_guess)
    half = max(3, int(box_size) // 2)
    stamp, sx0, sy0 = crop_around_xy(data, x0, y0, half)
    if stamp.size < 9 or np.sum(np.isfinite(stamp)) < 9:
        return coord_guess, x0, y0, "fallback_target_position"

    stamp2 = subtract_background(stamp)
    finite = np.isfinite(stamp2)
    if np.sum(finite) < 9:
        return coord_guess, x0, y0, "fallback_target_position"

    # Replace non-finite pixels by local median.
    med = np.nanmedian(stamp2[finite])
    stamp2 = np.where(finite, stamp2, med)

    try:
        if HAVE_PHOTUTILS:
            cyx = centroid_2dg(stamp2)
            cx, cy = scalar_float(cyx[0]), scalar_float(cyx[1])
            status = "ok_2dg"
        else:
            raise RuntimeError("photutils not available")
    except Exception:
        try:
            if HAVE_PHOTUTILS:
                cyx = centroid_com(stamp2)
                cx, cy = scalar_float(cyx[0]), scalar_float(cyx[1])
                status = "ok_com"
            else:
                # Simple center-of-light fallback after removing negative pedestal.
                img = stamp2 - np.nanpercentile(stamp2, 20)
                img = np.where(img > 0, img, 0)
                yy, xx = np.indices(img.shape)
                if np.nansum(img) <= 0:
                    raise RuntimeError("no positive flux")
                cx = float(np.nansum(xx * img) / np.nansum(img))
                cy = float(np.nansum(yy * img) / np.nansum(img))
                status = "ok_simple_com"
        except Exception:
            return coord_guess, x0, y0, "fallback_target_position"

    x = sx0 + cx
    y = sy0 + cy
    sky = wcs.pixel_to_world(x, y)
    return SkyCoord(sky), float(x), float(y), status


def make_cutout(data: np.ndarray, wcs: WCS, center: SkyCoord, size_arcsec: float) -> Cutout2D:
    size = u.Quantity((size_arcsec, size_arcsec), u.arcsec)
    return Cutout2D(data, position=center, size=size, wcs=wcs, mode="partial", fill_value=np.nan)


def mask_target_region(shape: Tuple[int, int], wcs: WCS, coord: SkyCoord, radius_arcsec: float) -> np.ndarray:
    yy, xx = np.indices(shape)
    x, y = world_to_pixel(wcs, coord)
    r_pix = radius_arcsec / pixel_scale_arcsec(wcs)
    return ((xx - x) ** 2 + (yy - y) ** 2) < r_pix ** 2


def robust_flux_scale(im1: np.ndarray, im2_aligned: np.ndarray, wcs: WCS, target_coord: SkyCoord) -> float:
    """
    Robust scale factor s such that s * im2_aligned approximately matches im1.
    Uses bright finite pixels outside the target region.
    """
    a = np.asarray(im1, dtype=float)
    b = np.asarray(im2_aligned, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    mask &= ~mask_target_region(a.shape, wcs, target_coord, TARGET_EXCLUSION_ARCSEC)

    if np.sum(mask) < 100:
        return 1.0

    av = a[mask]
    bv = b[mask]

    # Use moderately bright pixels to anchor sources, not blank sky.
    thresh_a = np.nanpercentile(av, 70)
    thresh_b = np.nanpercentile(bv, 70)
    m = (av > thresh_a) | (bv > thresh_b)
    av = av[m]
    bv = bv[m]

    if len(av) < 50 or np.nansum(bv ** 2) <= 0:
        return 1.0

    # Iterative robust slope.
    scale = float(np.nansum(av * bv) / np.nansum(bv ** 2))
    for _ in range(3):
        res = av - scale * bv
        med, sig = robust_median_sigma(res)
        good = np.isfinite(res) & (np.abs(res - med) < 3.0 * sig) if np.isfinite(sig) and sig > 0 else np.isfinite(res)
        if np.sum(good) < 20 or np.nansum(bv[good] ** 2) <= 0:
            break
        scale = float(np.nansum(av[good] * bv[good]) / np.nansum(bv[good] ** 2))

    if not np.isfinite(scale) or scale <= 0:
        return 1.0
    return scale


def align_epoch2_to_epoch1_grid(data2: np.ndarray, wcs2: WCS,
                                cut1: Cutout2D, ref_med_dra_mas: float,
                                ref_med_ddec_mas: float) -> np.ndarray:
    """
    Reproject epoch2 onto the epoch1 cutout WCS, then apply the residual image-level
    correction measured from reference sources.

    ref_med_dra/ref_med_ddec are the stationary-reference median offsets:
        epoch2 - epoch1.
    To align stationary sources, shift the reprojected epoch2 image by the negative
    of this vector in the epoch1 pixel grid.
    """
    if HAVE_REPROJECT:
        im2, _ = reproject_interp((data2, wcs2), cut1.wcs, shape_out=cut1.data.shape)
    else:
        # Fallback: make an epoch2 cutout centered at the same sky position.
        # This assumes both mosaics are already on almost the same north-up grid.
        center = cut1.wcs.pixel_to_world((cut1.data.shape[1] - 1) / 2, (cut1.data.shape[0] - 1) / 2)
        cut2_native = make_cutout(data2, wcs2, SkyCoord(center), FULL_SIZE_ARCSEC)
        im2 = cut2_native.data
        # Resize/crop if shape differs.
        if im2.shape != cut1.data.shape:
            out = np.full_like(cut1.data, np.nan, dtype=float)
            ny = min(out.shape[0], im2.shape[0])
            nx = min(out.shape[1], im2.shape[1])
            out[:ny, :nx] = im2[:ny, :nx]
            im2 = out

    if not HAVE_SCIPY:
        return im2

    p0 = np.array(world_to_pixel(cut1.wcs, TARGET), dtype=float)
    shifted_coord = skycoord_with_offsets(TARGET, ref_med_dra_mas, ref_med_ddec_mas)
    p_shifted = np.array(world_to_pixel(cut1.wcs, shifted_coord), dtype=float)
    delta_pix = p_shifted - p0

    # scipy shift uses (dy, dx).
    im2_aligned = ndi_shift(im2, shift=(-delta_pix[1], -delta_pix[0]), order=1,
                            mode="constant", cval=np.nan, prefilter=False)
    return im2_aligned


def normalize_image(data: np.ndarray, stretch: str = "asinh") -> ImageNormalize:
    arr = np.asarray(data, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 10:
        return ImageNormalize(vmin=0, vmax=1, stretch=LinearStretch())
    try:
        vmin, vmax = ZScaleInterval(contrast=0.25).get_limits(arr)
    except Exception:
        vmin, vmax = np.nanpercentile(arr, [1, 99])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanpercentile(arr, [1, 99])
    return ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch() if stretch == "asinh" else LinearStretch())


def diff_limits(diff: np.ndarray) -> Tuple[float, float]:
    arr = np.asarray(diff, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 10:
        return -1, 1
    vmax = np.nanpercentile(np.abs(arr), 99)
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = np.nanstd(arr) * 3
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1
    return -float(vmax), float(vmax)


def add_scale_bar(ax, wcs: WCS, length_arcsec: float = 2.0, color: str = "white") -> None:
    pix = length_arcsec / pixel_scale_arcsec(wcs)
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    xmin, xmax = min(x0, x1), max(x0, x1)
    ymin, ymax = min(y0, y1), max(y0, y1)
    x_start = xmax - pix - 0.07 * (xmax - xmin)
    y = ymin + 0.08 * (ymax - ymin)
    ax.plot([x_start, x_start + pix], [y, y], color=color, lw=2)
    ax.text(x_start + pix / 2, y + 0.03 * (ymax - ymin), f"{length_arcsec:.0f}″",
            color=color, ha="center", va="bottom", fontsize=9)


def add_position_circles(ax, wcs: WCS, coord1: SkyCoord, coord2_corr: SkyCoord,
                         radius_arcsec: float = APERTURE_RADIUS_ARCSEC) -> None:
    pixrad = radius_arcsec / pixel_scale_arcsec(wcs)
    x1, y1 = world_to_pixel(wcs, coord1)
    x2, y2 = world_to_pixel(wcs, coord2_corr)
    ax.add_patch(Circle((x1, y1), pixrad, fill=False, edgecolor="white", lw=1.8))
    ax.add_patch(Circle((x2, y2), pixrad, fill=False, edgecolor="red", lw=1.8))
    ax.plot(x1, y1, marker="+", color="white", ms=10, mew=1.8)
    ax.plot(x2, y2, marker="+", color="red", ms=10, mew=1.8)


def add_target_arrow(ax, wcs: WCS, coord: SkyCoord, text: str = "ID 639188",
                     color: str = "white") -> None:
    x, y = world_to_pixel(wcs, coord)
    ax.annotate(
        text, xy=(x, y), xytext=(x + 20, y + 12),
        color=color, fontsize=9, ha="left", va="center",
        arrowprops=dict(arrowstyle="->", color=color, lw=1.5)
    )


def set_pixel_axes_arcsec(ax, cut: Cutout2D, center_coord: SkyCoord) -> None:
    """
    Keep image pixel axes but label as relative arcsec from the target.
    """
    ny, nx = cut.data.shape
    x0, y0 = world_to_pixel(cut.wcs, center_coord)
    pixscale = pixel_scale_arcsec(cut.wcs)

    # Put about 5 ticks.
    xticks = np.linspace(0, nx - 1, 5)
    yticks = np.linspace(0, ny - 1, 5)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xticklabels([f"{(x - x0) * pixscale:.1f}" for x in xticks])
    ax.set_yticklabels([f"{(y - y0) * pixscale:.1f}" for y in yticks])
    ax.set_xlabel("Δx [arcsec]")
    ax.set_ylabel("Δy [arcsec]")


def proper_motion_pa_deg(mu_ra_masyr: float, mu_dec_masyr: float) -> float:
    """
    Position angle east of north: 0=north, 90=east.
    Uses μ_RA*cosDec and μ_Dec.
    """
    pa = np.degrees(np.arctan2(mu_ra_masyr, mu_dec_masyr))
    return float((pa + 360) % 360)


def add_motion_arrow(ax, wcs: WCS, coord1: SkyCoord, dra_mas: float, ddec_mas: float,
                     color: str = "red") -> None:
    coord2 = skycoord_with_offsets(coord1, dra_mas, ddec_mas)
    x1, y1 = world_to_pixel(wcs, coord1)
    x2, y2 = world_to_pixel(wcs, coord2)
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=color, lw=2.0))


# =============================================================================
# FIGURE MAKER
# =============================================================================

def make_one_fig3_style_page(row: pd.Series) -> Tuple[plt.Figure, dict]:
    name1 = str(row["epoch1"])
    name2 = str(row["epoch2"])

    if name1 not in IMAGE_EPOCHS or name2 not in IMAGE_EPOCHS:
        raise KeyError(f"Pair uses epoch names not in IMAGE_EPOCHS: {name1}, {name2}")

    filt1 = filter_from_epoch_name(name1)
    filt2 = filter_from_epoch_name(name2)
    if filt1 != filt2:
        print(f"WARNING: {name1}->{name2} are not the same filter ({filt1}, {filt2}). Difference image may be misleading.")

    data1, wcs1, _ = load_image(name1)
    data2, wcs2, _ = load_image(name2)

    # Candidate centroid in each native image, using target coordinate as initial guess.
    coord1_meas, x1_native, y1_native, status1 = centroid_source(data1, wcs1, TARGET)
    coord2_meas, x2_native, y2_native, status2 = centroid_source(data2, wcs2, TARGET)

    # Use the corrected displacement from your previous image-level registration.
    dra_corr = float(row["cand_corr_dra_mas"])
    ddec_corr = float(row["cand_corr_ddec_mas"])
    coord2_corr = skycoord_with_offsets(coord1_meas, dra_corr, ddec_corr)

    ref_dra = float(row.get("ref_med_dra_mas", 0.0))
    ref_ddec = float(row.get("ref_med_ddec_mas", 0.0))

    # Full-field cutout in epoch1 frame.
    cut1_full = make_cutout(data1, wcs1, TARGET, FULL_SIZE_ARCSEC)
    im2_full_aligned = align_epoch2_to_epoch1_grid(data2, wcs2, cut1_full, ref_dra, ref_ddec)

    scale = robust_flux_scale(cut1_full.data, im2_full_aligned, cut1_full.wcs, TARGET) if AUTO_FLUX_SCALE_FOR_DIFF else 1.0
    im2_full_scaled = scale * im2_full_aligned
    diff_full = im2_full_scaled - cut1_full.data

    # Zoom cutouts from the full-field already-aligned arrays, centered on the target.
    # This guarantees the same pixel frame in all three zoom panels.
    cut1_zoom = make_cutout(cut1_full.data, cut1_full.wcs, TARGET, ZOOM_SIZE_ARCSEC)
    cut2_zoom = make_cutout(im2_full_scaled, cut1_full.wcs, TARGET, ZOOM_SIZE_ARCSEC)
    cutd_zoom = make_cutout(diff_full, cut1_full.wcs, TARGET, ZOOM_SIZE_ARCSEC)

    # Proper-motion diagnostics.
    dt_yr = float(row["dt_yr"])
    mu_ra = float(row["mu_ra_masyr"])
    mu_dec = float(row["mu_dec_masyr"])
    mu_tot = float(row["mu_total_masyr"])
    pa = proper_motion_pa_deg(mu_ra, mu_dec)

    # Approximate diagnostic uncertainties from image-level registration scatter.
    # This does NOT replace fake-source injection.
    n_ref = max(float(row.get("n_ref", np.nan)), 1.0)
    sig_dra_reg = float(row.get("ref_sig_dra_mas", np.nan)) / math.sqrt(n_ref)
    sig_ddec_reg = float(row.get("ref_sig_ddec_mas", np.nan)) / math.sqrt(n_ref)
    sig_tot_mas = math.sqrt(sig_dra_reg ** 2 + sig_ddec_reg ** 2) if np.isfinite(sig_dra_reg) and np.isfinite(sig_ddec_reg) else np.nan
    sig_mu_tot = sig_tot_mas / dt_yr if np.isfinite(sig_tot_mas) and dt_yr > 0 else np.nan

    # -------------------------------------------------------------------------
    # Plot: top row full cutout, bottom row zoom cutout.
    # -------------------------------------------------------------------------
    fig = plt.figure(figsize=(11.0, 7.4))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.25, 1.0], hspace=0.22, wspace=0.08)

    ax00 = fig.add_subplot(gs[0, 0])
    ax01 = fig.add_subplot(gs[0, 1])
    ax02 = fig.add_subplot(gs[0, 2])
    ax10 = fig.add_subplot(gs[1, 0])
    ax11 = fig.add_subplot(gs[1, 1])
    ax12 = fig.add_subplot(gs[1, 2])

    image_axes = [ax00, ax01, ax10, ax11]
    diff_axes = [ax02, ax12]

    # Full panels
    ax00.imshow(cut1_full.data, origin="lower", cmap="gray", norm=normalize_image(cut1_full.data))
    ax01.imshow(im2_full_scaled, origin="lower", cmap="gray", norm=normalize_image(im2_full_scaled))
    vmin, vmax = diff_limits(diff_full)
    ax02.imshow(diff_full, origin="lower", cmap="RdBu_r", vmin=vmin, vmax=vmax)

    ax00.set_title(f"a) {IMAGE_EPOCHS[name1]['label']}", fontsize=11)
    ax01.set_title(f"b) {IMAGE_EPOCHS[name2]['label']} aligned", fontsize=11)
    ax02.set_title("c) Difference image: epoch 2 − epoch 1", fontsize=11)

    for ax in [ax00, ax01, ax02]:
        add_target_arrow(ax, cut1_full.wcs, coord1_meas, text="ID 639188", color="white" if ax is not ax02 else "black")
        add_scale_bar(ax, cut1_full.wcs, 2.0, color="white" if ax is not ax02 else "black")
        ax.set_xticks([])
        ax.set_yticks([])

    # Zoom panels
    ax10.imshow(cut1_zoom.data, origin="lower", cmap="viridis", norm=normalize_image(cut1_zoom.data))
    ax11.imshow(cut2_zoom.data, origin="lower", cmap="viridis", norm=normalize_image(cut2_zoom.data))
    vminz, vmaxz = diff_limits(cutd_zoom.data)
    ax12.imshow(cutd_zoom.data, origin="lower", cmap="RdBu_r", vmin=vminz, vmax=vmaxz)

    ax10.set_title(f"Epoch 1 zoom ({filt1})", fontsize=10)
    ax11.set_title(f"Epoch 2 zoom ({filt2})", fontsize=10)
    ax12.set_title("Difference zoom", fontsize=10)

    for ax, cut in [(ax10, cut1_zoom), (ax11, cut2_zoom), (ax12, cutd_zoom)]:
        add_position_circles(ax, cut.wcs, coord1_meas, coord2_corr, APERTURE_RADIUS_ARCSEC)
        add_motion_arrow(ax, cut.wcs, coord1_meas, dra_corr, ddec_corr, color="red")
        set_pixel_axes_arcsec(ax, cut, TARGET)

    # Cleaner tick labels for middle/right zoom panels.
    ax11.set_ylabel("")
    ax12.set_ylabel("")
    ax11.set_yticklabels([])
    ax12.set_yticklabels([])

    subtitle = (
        f"{name1} → {name2}:  Δt={dt_yr:.3f} yr, "
        f"μ=({mu_ra:.1f}, {mu_dec:.1f}) mas/yr, "
        f"|μ|={mu_tot:.1f}"
    )
    if np.isfinite(sig_mu_tot):
        subtitle += f" ± {sig_mu_tot:.1f}"
    subtitle += f" mas/yr, PA={pa:.0f}° E of N"

    fig.suptitle(
        f"COSMOS2025 ID {OBJID}: Fig. 3-style proper-motion check\n"
        + subtitle,
        fontsize=13,
        y=0.985,
    )

    # Legend-like note.
    #fig.text(
    #    0.5, 0.012,
     #   "White circle/cross: epoch-1 centroid. Red circle/cross/arrow: epoch-2 centroid after image-level registration. "
     #   f"Diff scaled by factor {scale:.3f}. Red=positive, blue=negative.",
     #   ha="center",
     #   fontsize=9,
    #)

    diag = {
        "epoch1": name1,
        "epoch2": name2,
        "filter1": filt1,
        "filter2": filt2,
        "dt_yr": dt_yr,
        "centroid_status_epoch1_remeasured": status1,
        "centroid_status_epoch2_remeasured": status2,
        "x1_native": x1_native,
        "y1_native": y1_native,
        "x2_native": x2_native,
        "y2_native": y2_native,
        "dra_corr_mas": dra_corr,
        "ddec_corr_mas": ddec_corr,
        "mu_ra_masyr": mu_ra,
        "mu_dec_masyr": mu_dec,
        "mu_total_masyr": mu_tot,
        "pa_deg_east_of_north": pa,
        "diff_flux_scale_epoch2_to_epoch1": scale,
        "diagnostic_sigma_mu_total_masyr": sig_mu_tot,
    }
    return fig, diag


def choose_pairs(summary: pd.DataFrame) -> pd.DataFrame:
    s = summary.copy()
    if PAIRS_TO_PLOT is None:
        pass
    else:
        wanted = set(PAIRS_TO_PLOT)
        s = s[s.apply(lambda r: (r["epoch1"], r["epoch2"]) in wanted, axis=1)]

    # Require known image entries and existing files.
    keep = []
    for _, r in s.iterrows():
        e1, e2 = str(r["epoch1"]), str(r["epoch2"])
        if e1 not in IMAGE_EPOCHS or e2 not in IMAGE_EPOCHS:
            keep.append(False)
            continue
        if not Path(IMAGE_EPOCHS[e1]["cutout"]).exists():
            print(f"Skipping {e1}->{e2}: missing {IMAGE_EPOCHS[e1]['cutout']}")
            keep.append(False)
            continue
        if not Path(IMAGE_EPOCHS[e2]["cutout"]).exists():
            print(f"Skipping {e1}->{e2}: missing {IMAGE_EPOCHS[e2]['cutout']}")
            keep.append(False)
            continue
        keep.append(True)
    s = s[np.array(keep, dtype=bool)].reset_index(drop=True)

    # Long-baseline pairs first, then by filter.
    if len(s):
        s["_filter"] = s["epoch1"].apply(filter_from_epoch_name)
        s = s.sort_values(["dt_yr", "_filter"], ascending=[False, True]).drop(columns=["_filter"])
    return s.reset_index(drop=True)


def save_pm_summary_plots(summary: pd.DataFrame) -> None:
    if summary.empty:
        return

    # Vector plot: corrected displacement, with reference cloud already shown in your old diagnostic.
    fig, ax = plt.subplots(figsize=(6.6, 6.2))
    for _, r in summary.iterrows():
        x = float(r["cand_corr_dra_mas"])
        y = float(r["cand_corr_ddec_mas"])
        lab = f"{filter_from_epoch_name(str(r['epoch1']))}: {str(r['epoch1']).split('_')[0]}→{str(r['epoch2']).split('_')[0]}"
        ax.scatter(x, y, s=80)
        ax.text(x + 3, y + 3, lab, fontsize=8)

    ax.axhline(0, color="0.5", lw=1)
    ax.axvline(0, color="0.5", lw=1)
    lim = max(30, np.nanmax(np.abs(summary[["cand_corr_dra_mas", "cand_corr_ddec_mas"]].to_numpy(float))) * 1.25)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Corrected ΔRA [mas]")
    ax.set_ylabel("Corrected ΔDec [mas]")
    ax.set_title(f"ID {OBJID}: corrected displacement in available epochs")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    out = OUTDIR / f"ID{OBJID}_available_epoch_displacements.png"
    fig.savefig(out, dpi=250)
    plt.close(fig)
    print("Saved:", out)

    # Proper-motion by pair.
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    x = np.arange(len(summary))
    labels = [f"{filter_from_epoch_name(str(r['epoch1']))}\n{r['dt_yr']:.2f} yr" for _, r in summary.iterrows()]
    ax.scatter(x, summary["mu_total_masyr"].to_numpy(float), s=80)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("|μ| [mas/yr]")
    ax.set_title(f"ID {OBJID}: proper motion from each epoch pair")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    out = OUTDIR / f"ID{OBJID}_mu_by_epoch_pair.png"
    fig.savefig(out, dpi=250)
    plt.close(fig)
    print("Saved:", out)


def main() -> None:
    summary_path = find_summary_csv()
    summary = pd.read_csv(summary_path)
    print(f"Loaded image-level summary: {summary_path}")
    print(summary.to_string(index=False))

    pairs = choose_pairs(summary)
    if pairs.empty:
        raise RuntimeError(
            "No plottable pairs found. Check CUTOUT_DIR and IMAGE_EPOCHS paths."
        )

    print("\nPairs to plot:")
    print(pairs[["epoch1", "epoch2", "dt_yr", "mu_total_masyr"]].to_string(index=False))

    diag_rows = []
    pdf_path = OUTDIR / f"ID{OBJID}_Fig3_style_all_available_epoch_pairs.pdf"
    with PdfPages(pdf_path) as pdf:
        for _, row in pairs.iterrows():
            fig, diag = make_one_fig3_style_page(row)
            diag_rows.append(diag)

            e1 = str(row["epoch1"])
            e2 = str(row["epoch2"])
            png_path = OUTDIR / f"ID{OBJID}_Fig3_style_{e1}_to_{e2}.png"
            fig.savefig(png_path, dpi=250, bbox_inches="tight")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            print("Saved:", png_path)

    print("Saved:", pdf_path)

    diag = pd.DataFrame(diag_rows)
    diag_path = OUTDIR / f"ID{OBJID}_Fig3_style_diagnostics.csv"
    diag.to_csv(diag_path, index=False)
    print("Saved:", diag_path)

    save_pm_summary_plots(pairs)

    # A compact recommendation for which pairs are strongest.
    long_pairs = pairs[pairs["dt_yr"] >= MIN_BASELINE_FOR_MAIN_LABEL_YR].copy()
    if not long_pairs.empty:
        print("\nLong-baseline same-filter measurements are the best to emphasize:")
        cols = ["epoch1", "epoch2", "dt_yr", "mu_ra_masyr", "mu_dec_masyr", "mu_total_masyr", "n_ref"]
        print(long_pairs[cols].to_string(index=False))

    print("\nDone.")


if __name__ == "__main__":
    main()


Loaded image-level summary: proper_motion_image_level/image_level_pm_summary.csv
              epoch1               epoch2    dt_yr  n_ref  ref_med_dra_mas  ref_med_ddec_mas  ref_sig_dra_mas  ref_sig_ddec_mas  cand_raw_dra_mas  cand_raw_ddec_mas  cand_corr_dra_mas  cand_corr_ddec_mas  cand_total_mas  mu_ra_masyr  mu_dec_masyr  mu_total_masyr cand_status_epoch1 cand_status_epoch2
   PRIMER_2023_F115W COSMOSWeb_2023_F115W 0.676249     47        17.762695         -4.539466         3.976022          7.248696        -18.463912         -47.056991         -36.226607          -42.517524       55.857917   -53.569912    -62.872574       82.599613             ok_2dg             ok_2dg
COSMOSWeb_2023_F115W  COSMOS3D_2025_F115W 1.292266     36       -14.349322         18.294537         4.497685         17.271762       -112.190589         -43.568972         -97.841267          -61.863508      115.758400   -75.712972    -47.872132       89.577872             ok_2dg             ok_2dg
   PRIMER_2023_F

Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729580.833 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/ID639188_Fig3_style_PRIMER_2023_F115W_to_COSMOS3D_2025_F115W.png


Set OBSGEO-B to   -13.421398 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412384141.197 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.701718 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416390764.218 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/ID639188_Fig3_style_PRIMER_2023_F200W_to_COSMOS3D_2025_F200W.png


Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729589.007 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/ID639188_Fig3_style_PRIMER_2023_F356W_to_COSMOS3D_2025_F356W.png


Set OBSGEO-B to     8.880265 from OBSGEO-[XYZ].
Set OBSGEO-H to 1695430968.518 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729580.833 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/ID639188_Fig3_style_COSMOSWeb_2023_F115W_to_COSMOS3D_2025_F115W.png


Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to     8.880265 from OBSGEO-[XYZ].
Set OBSGEO-H to 1695430968.518 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/ID639188_Fig3_style_PRIMER_2023_F115W_to_COSMOSWeb_2023_F115W.png
Saved: ID639188_Fig3_style_PM/ID639188_Fig3_style_all_available_epoch_pairs.pdf
Saved: ID639188_Fig3_style_PM/ID639188_Fig3_style_diagnostics.csv
Saved: ID639188_Fig3_style_PM/ID639188_available_epoch_displacements.png
Saved: ID639188_Fig3_style_PM/ID639188_mu_by_epoch_pair.png

Long-baseline same-filter measurements are the best to emphasize:
              epoch1              epoch2    dt_yr  mu_ra_masyr  mu_dec_masyr  mu_total_masyr  n_ref
   PRIMER_2023_F115W COSMOS3D_2025_F115W 1.968515   -68.609866    -52.936058       86.657602     47
   PRIMER_2023_F200W COSMOS3D_2025_F200W 1.968515   -68.016251    -54.109282       86.913893     79
   PRIMER_2023_F356W COSMOS3D_2025_F356W 1.968515   -71.249379    -53.074086       88.844429     81
COSMOSWeb_2023_F115W COSMOS3D_2025_F115W 1.292266   -75.712972    -47.872132       89.577872     36

Done.


In [4]:

# =============================================================================
# PAPER-FRIENDLY COMBINED PROPER-MOTION FIGURES
# Run this cell AFTER the main proper_motion_fig3.ipynb cell.
# It reuses the notebook functions and the same FITS cutouts / summary CSV.
# =============================================================================

from pathlib import Path
import math
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from astropy.visualization import ImageNormalize, AsinhStretch


# Longest, independent same-filter baselines: recommended for the main paper.
MAIN_PAIRS = [
    ("PRIMER_2023_F115W", "COSMOS3D_2025_F115W"),
    ("PRIMER_2023_F200W", "COSMOS3D_2025_F200W"),
    ("PRIMER_2023_F356W", "COSMOS3D_2025_F356W"),
]

# Shorter F115W checks: recommended for the appendix.
APPENDIX_PAIRS = [
    ("PRIMER_2023_F115W", "COSMOSWeb_2023_F115W"),
    ("COSMOSWeb_2023_F115W", "COSMOS3D_2025_F115W"),
]

PAPER_OUTDIR = OUTDIR / "paper_friendly"
PAPER_OUTDIR.mkdir(parents=True, exist_ok=True)

# Zoom extent. Keep the same physical field in every panel.
PAPER_ZOOM_SIZE_ARCSEC = 1.8


def _get_pair_row(summary: pd.DataFrame, epoch1: str, epoch2: str) -> pd.Series:
    """Return one exact epoch-pair row from the image-level summary."""
    m = (summary["epoch1"].astype(str) == epoch1) & (summary["epoch2"].astype(str) == epoch2)
    if m.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {epoch1} -> {epoch2}, but found {m.sum()}."
        )
    return summary.loc[m].iloc[0]


def _diagnostic_sigma_mu(row: pd.Series) -> float:
    """
    Reproduce the image-registration diagnostic uncertainty used by the
    original plotting notebook. Replace this with the final fake-source
    uncertainty column when that result is frozen.
    """
    # Prefer a final uncertainty column if one has been added to the CSV.
    for col in [
        "mu_total_err_masyr",
        "sigma_mu_total_masyr",
        "fake_sigma_mu_total_masyr",
        "moving_fake_sigma_masyr",
    ]:
        if col in row.index:
            value = float(row[col])
            if np.isfinite(value) and value > 0:
                return value

    n_ref = max(float(row.get("n_ref", np.nan)), 1.0)
    sig_dra = float(row.get("ref_sig_dra_mas", np.nan)) / np.sqrt(n_ref)
    sig_ddec = float(row.get("ref_sig_ddec_mas", np.nan)) / np.sqrt(n_ref)
    dt_yr = float(row["dt_yr"])

    if np.isfinite(sig_dra) and np.isfinite(sig_ddec) and dt_yr > 0:
        return np.hypot(sig_dra, sig_ddec) / dt_yr
    return np.nan


def _shared_image_norm(image1: np.ndarray, image2: np.ndarray) -> ImageNormalize:
    """
    One common display normalization for the two registered epochs in a row.
    This makes the visual comparison more honest than separately stretching
    each epoch.
    """
    values = np.concatenate(
        [
            np.asarray(image1, dtype=float).ravel(),
            np.asarray(image2, dtype=float).ravel(),
        ]
    )
    values = values[np.isfinite(values)]

    if values.size < 20:
        return ImageNormalize(vmin=0.0, vmax=1.0, stretch=AsinhStretch())

    vmin, vmax = np.nanpercentile(values, [1.0, 99.5])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = np.nanmin(values), np.nanmax(values)

    return ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch())


def _prepare_zoom_triplet(row: pd.Series) -> dict:
    """
    Reproduce the registration and subtraction, but retain only the compact
    target-centered zoom panels needed for the paper figure.
    """
    name1 = str(row["epoch1"])
    name2 = str(row["epoch2"])

    data1, wcs1, _ = load_image(name1)
    data2, wcs2, _ = load_image(name2)

    # Re-measure the target centroid in the native epoch-1 image.
    coord1_meas, _, _, status1 = centroid_source(data1, wcs1, TARGET)
    _, _, _, status2 = centroid_source(data2, wcs2, TARGET)

    # Corrected displacement from the image-level registration analysis.
    dra_corr = float(row["cand_corr_dra_mas"])
    ddec_corr = float(row["cand_corr_ddec_mas"])
    coord2_corr = skycoord_with_offsets(coord1_meas, dra_corr, ddec_corr)

    ref_dra = float(row.get("ref_med_dra_mas", 0.0))
    ref_ddec = float(row.get("ref_med_ddec_mas", 0.0))

    # A sufficiently large common frame is used for image registration.
    cut1_full = make_cutout(data1, wcs1, TARGET, FULL_SIZE_ARCSEC)
    image2_aligned = align_epoch2_to_epoch1_grid(
        data2, wcs2, cut1_full, ref_dra, ref_ddec
    )

    scale = (
        robust_flux_scale(
            cut1_full.data, image2_aligned, cut1_full.wcs, TARGET
        )
        if AUTO_FLUX_SCALE_FOR_DIFF
        else 1.0
    )
    image2_scaled = scale * image2_aligned
    difference = image2_scaled - cut1_full.data

    # All three panels now share exactly the same WCS and physical footprint.
    cut1 = make_cutout(
        cut1_full.data, cut1_full.wcs, TARGET, PAPER_ZOOM_SIZE_ARCSEC
    )
    cut2 = make_cutout(
        image2_scaled, cut1_full.wcs, TARGET, PAPER_ZOOM_SIZE_ARCSEC
    )
    cutd = make_cutout(
        difference, cut1_full.wcs, TARGET, PAPER_ZOOM_SIZE_ARCSEC
    )

    return {
        "row": row,
        "name1": name1,
        "name2": name2,
        "filter": filter_from_epoch_name(name1),
        "cut1": cut1,
        "cut2": cut2,
        "cutd": cutd,
        "coord1": coord1_meas,
        "coord2": coord2_corr,
        "dra_corr": dra_corr,
        "ddec_corr": ddec_corr,
        "scale": scale,
        "status1": status1,
        "status2": status2,
    }


def _add_clean_centroids_and_arrow(
    ax,
    wcs,
    coord1,
    coord2,
    arrow_color="tab:red",
):
    """
    Minimal overlay:
      + = epoch 1
      x = epoch 2
      arrow = epoch-1 to epoch-2 displacement

    Marker shape, not only color, distinguishes the two epochs.
    """
    x1, y1 = world_to_pixel(wcs, coord1)
    x2, y2 = world_to_pixel(wcs, coord2)

    outline = [pe.Stroke(linewidth=2.6, foreground="black"), pe.Normal()]

    p1 = ax.plot(
        x1, y1, marker="+", ms=8.0, mew=1.6,
        color="white", linestyle="none", zorder=6
    )[0]
    p1.set_path_effects(outline)

    p2 = ax.plot(
        x2, y2, marker="x", ms=7.0, mew=1.6,
        color=arrow_color, linestyle="none", zorder=7
    )[0]
    p2.set_path_effects(outline)

    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle="-|>",
            color=arrow_color,
            lw=1.4,
            mutation_scale=8,
            shrinkA=1.5,
            shrinkB=1.5,
        ),
        zorder=5,
    )


def _format_relative_axes(ax, cut, show_xlabel: bool, show_ylabel: bool) -> None:
    """Use the same relative-arcsecond axes, but suppress repeated labels."""
    set_pixel_axes_arcsec(ax, cut, TARGET)

    ax.tick_params(
        axis="both",
        which="both",
        direction="in",
        top=True,
        right=True,
        length=2.5,
        pad=2,
    )

    if not show_xlabel:
        ax.set_xlabel("")
        ax.set_xticklabels([])

    if not show_ylabel:
        ax.set_ylabel("")
        ax.set_yticklabels([])


def make_combined_pm_figure(
    pair_list,
    output_stem: str,
    row_label_mode: str = "full",
):
    """
    Create a compact N-row x 3-column figure:
        column 1: epoch 1
        column 2: aligned/scaled epoch 2
        column 3: difference image

    Parameters
    ----------
    pair_list : list of (epoch1, epoch2)
    output_stem : output filename without extension
    row_label_mode :
        "full"  -> filter, baseline, and |mu| in the left margin
        "short" -> filter and |mu| only
    """
    summary = pd.read_csv(find_summary_csv())
    rows = [_get_pair_row(summary, e1, e2) for e1, e2 in pair_list]
    prepared = [_prepare_zoom_triplet(row) for row in rows]

    nrows = len(prepared)

    # AASTeX two-column full-width figure is about 7.1 inches wide.
    fig_height = 1.82 * nrows + 0.62

    style = {
        "font.size": 8.0,
        "axes.titlesize": 9.0,
        "axes.labelsize": 8.0,
        "xtick.labelsize": 7.0,
        "ytick.labelsize": 7.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.dpi": 300,
    }

    with plt.rc_context(style):
        fig, axes = plt.subplots(
            nrows,
            3,
            figsize=(7.15, fig_height),
            squeeze=False,
            gridspec_kw={
                "left": 0.16,
                "right": 0.995,
                "bottom": 0.08,
                "top": 0.94,
                "wspace": 0.035,
                "hspace": 0.085,
            },
        )

        column_titles = [
            "Epoch 1",
            "Epoch 2 aligned",
            r"Epoch 2 $-$ Epoch 1",
        ]
        for j, title in enumerate(column_titles):
            axes[0, j].set_title(title, pad=4)

        panel_index = 0

        for i, item in enumerate(prepared):
            row = item["row"]
            cut1 = item["cut1"]
            cut2 = item["cut2"]
            cutd = item["cutd"]

            norm = _shared_image_norm(cut1.data, cut2.data)
            dmin, dmax = diff_limits(cutd.data)

            axes[i, 0].imshow(
                cut1.data,
                origin="lower",
                cmap="gray",
                norm=norm,
                interpolation="nearest",
            )
            axes[i, 1].imshow(
                cut2.data,
                origin="lower",
                cmap="gray",
                norm=norm,
                interpolation="nearest",
            )
            axes[i, 2].imshow(
                cutd.data,
                origin="lower",
                cmap="RdBu_r",
                vmin=dmin,
                vmax=dmax,
                interpolation="nearest",
            )

            for j, cut in enumerate([cut1, cut2, cutd]):
                _add_clean_centroids_and_arrow(
                    axes[i, j],
                    cut.wcs,
                    item["coord1"],
                    item["coord2"],
                )
                _format_relative_axes(
                    axes[i, j],
                    cut,
                    show_xlabel=(i == nrows - 1),
                    show_ylabel=(j == 0),
                )

                letter = string.ascii_lowercase[panel_index]
                axes[i, j].text(
                    0.035,
                    0.955,
                    f"({letter})",
                    transform=axes[i, j].transAxes,
                    ha="left",
                    va="top",
                    fontsize=8.0,
                    fontweight="bold",
                    color="black",
                    bbox=dict(
                        boxstyle="round,pad=0.15",
                        facecolor="white",
                        edgecolor="none",
                        alpha=0.78,
                    ),
                    zorder=10,
                )
                panel_index += 1

            dt = float(row["dt_yr"])
            mu_ra = float(row["mu_ra_masyr"])
            mu_dec = float(row["mu_dec_masyr"])
            mu_tot = float(row["mu_total_masyr"])
            sigma_mu = _diagnostic_sigma_mu(row)

            if row_label_mode == "short":
                label = (
                    f"{item['filter']}\n"
                    rf"$|\mu|={mu_tot:.1f}$ mas yr$^{{-1}}$"
                )
            else:
                label = (
                    f"{item['filter']}\n"
                    rf"$\Delta t={dt:.3f}$ yr" "\n"
                    rf"$\mu_{{\alpha *}}={mu_ra:.1f}$" "\n"
                    rf"$\mu_\delta={mu_dec:.1f}$" "\n"
                    rf"$|\mu|={mu_tot:.1f}$"
                )
                if np.isfinite(sigma_mu):
                    label += rf"$\pm{sigma_mu:.1f}$"
                label += r" mas yr$^{-1}$"

            axes[i, 0].text(
                -0.47,
                0.50,
                label,
                transform=axes[i, 0].transAxes,
                ha="center",
                va="center",
                fontsize=7.6,
                linespacing=1.18,
                clip_on=False,
            )

        # A single legend replaces the repeated sentence below every old figure.
        fig.text(
            0.575,
            0.018,
            r"$+$ epoch 1 centroid; $\times$ epoch 2 centroid; arrow shows the measured displacement. "
            r"Difference panels use a symmetric red/blue scale.",
            ha="center",
            va="bottom",
            fontsize=7.2,
        )

        pdf_path = PAPER_OUTDIR / f"{output_stem}.pdf"
        png_path = PAPER_OUTDIR / f"{output_stem}.png"

        fig.savefig(pdf_path, bbox_inches="tight")
        fig.savefig(png_path, dpi=350, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    print("Saved:", pdf_path)
    print("Saved:", png_path)
    return pdf_path, png_path


# -------------------------------------------------------------------------
# 1. Recommended main-text figure: three independent 1.969-yr baselines.
# -------------------------------------------------------------------------
main_pdf, main_png = make_combined_pm_figure(
    MAIN_PAIRS,
    output_stem=f"ID{OBJID}_proper_motion_main_3x3",
    row_label_mode="full",
)

# -------------------------------------------------------------------------
# 2. Recommended appendix figure: two shorter F115W consistency checks.
# -------------------------------------------------------------------------
appendix_pdf, appendix_png = make_combined_pm_figure(
    APPENDIX_PAIRS,
    output_stem=f"ID{OBJID}_proper_motion_appendix_2x3",
    row_label_mode="full",
)

# -------------------------------------------------------------------------
# 3. Optional: all five comparisons in one figure.
#    This is useful for an appendix or referee response, but is probably too
#    dense for the main text.
# -------------------------------------------------------------------------
ALL_PAIRS = MAIN_PAIRS + APPENDIX_PAIRS
all_pdf, all_png = make_combined_pm_figure(
    ALL_PAIRS,
    output_stem=f"ID{OBJID}_proper_motion_all_pairs_5x3",
    row_label_mode="short",
)


Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729580.833 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.421398 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412384141.197 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.701718 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416390764.218 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729589.007 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/paper_friendly/ID639188_proper_motion_main_3x3.pdf
Saved: ID639188_Fig3_style_PM/paper_friendly/ID639188_proper_motion_main_3x3.png


Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to     8.880265 from OBSGEO-[XYZ].
Set OBSGEO-H to 1695430968.518 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to     8.880265 from OBSGEO-[XYZ].
Set OBSGEO-H to 1695430968.518 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729580.833 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/paper_friendly/ID639188_proper_motion_appendix_2x3.pdf
Saved: ID639188_Fig3_style_PM/paper_friendly/ID639188_proper_motion_appendix_2x3.png


Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729580.833 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.421398 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412384141.197 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.701718 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416390764.218 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729589.007 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to     8.880265 from OBSGEO-[XYZ].
Set OBSGEO-H to 1695430968.518 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to     8.880265 from OBSGEO-[XYZ].
Set OBSGEO-H to 

Saved: ID639188_Fig3_style_PM/paper_friendly/ID639188_proper_motion_all_pairs_5x3.pdf
Saved: ID639188_Fig3_style_PM/paper_friendly/ID639188_proper_motion_all_pairs_5x3.png


In [11]:

# =============================================================================
# FINAL PAPER FIGURES FOR THE PROPER-MOTION SECTION
#
# Run this cell after the first cell of proper_motion_fig3.ipynb.
#
# Output 1: a compact 3 x 3 image-registration figure using only the three
#           longest, independent PRIMER -> COSMOS-3D same-filter baselines.
# Output 2: a quantitative two-panel astrometric summary with fake-source
#           recovery error bars.
#
# No image smoothing is applied. imshow(interpolation="none") preserves the
# native pixel structure. Epoch images use one shared colour normalization per
# row; difference images use a symmetric diverging scale.
# =============================================================================

from pathlib import Path
import math
import string
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.gridspec import GridSpec
from astropy.visualization import ImageNormalize, AsinhStretch


# -----------------------------------------------------------------------------
# Pair selection
# -----------------------------------------------------------------------------

# Main-text image figure: the three longest and independent same-filter pairs.
MAIN_PAIRS = [
    ("PRIMER_2023_F115W", "COSMOS3D_2025_F115W"),
    ("PRIMER_2023_F200W", "COSMOS3D_2025_F200W"),
    ("PRIMER_2023_F356W", "COSMOS3D_2025_F356W"),
]

# Quantitative figure: add the independent COSMOS-Web -> COSMOS-3D F115W check.
# The short PRIMER -> COSMOS-Web F115W pair is deliberately omitted from the
# main figures because its short baseline makes it mainly a sanity check.
QUANTITATIVE_PAIRS = MAIN_PAIRS + [
    ("COSMOSWeb_2023_F115W", "COSMOS3D_2025_F115W"),
]

SURVEY_SHORT = {
    "PRIMER": "PRIMER",
    "COSMOSWeb": "COSMOS-Web",
    "COSMOS3D": "COSMOS-3D",
}

PAIR_DISPLAY = {
    ("PRIMER_2023_F115W", "COSMOS3D_2025_F115W"):
        "F115W: PRIMER → COSMOS-3D",
    ("PRIMER_2023_F200W", "COSMOS3D_2025_F200W"):
        "F200W: PRIMER → COSMOS-3D",
    ("PRIMER_2023_F356W", "COSMOS3D_2025_F356W"):
        "F356W: PRIMER → COSMOS-3D",
    ("COSMOSWeb_2023_F115W", "COSMOS3D_2025_F115W"):
        "F115W: COSMOS-Web → COSMOS-3D",
}

# Use the fake-source moving-recovery output for manuscript uncertainties.
FAKE_SUMMARY_CANDIDATES = [
    Path("ID639188_fake_source_PM_test/ID639188_fake_source_recovery_summary.csv"),
    Path("ID639188_fake_source_recovery_summary.csv"),
    Path("/mnt/data/ID639188_fake_source_recovery_summary.csv"),
]

COMBINED_SUMMARY_CANDIDATES = [
    Path("ID639188_fake_source_PM_test/ID639188_fake_source_combined_pm_summary.csv"),
    Path("ID639188_fake_source_combined_pm_summary.csv"),
    Path("/mnt/data/ID639188_fake_source_combined_pm_summary.csv"),
]

PAPER_OUTDIR = OUTDIR / "paper_ready"
PAPER_OUTDIR.mkdir(parents=True, exist_ok=True)

PAPER_ZOOM_SIZE_ARCSEC = 1.8
SOURCE_CMAP = "magma"
DIFFERENCE_CMAP = "RdBu_r"

# Set this to False only for debugging. A final manuscript figure should use
# fake-source uncertainties rather than registration-only diagnostic errors.
REQUIRE_FAKE_SOURCE_ERRORS = True


# -----------------------------------------------------------------------------
# Data access and uncertainty helpers
# -----------------------------------------------------------------------------

def _first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


def _get_pair_row(summary: pd.DataFrame, epoch1: str, epoch2: str) -> pd.Series:
    match = (
        (summary["epoch1"].astype(str) == epoch1)
        & (summary["epoch2"].astype(str) == epoch2)
    )
    if match.sum() != 1:
        raise ValueError(
            f"Expected one row for {epoch1} -> {epoch2}; found {match.sum()}."
        )
    return summary.loc[match].iloc[0]


def _load_fake_moving_summary() -> pd.DataFrame:
    path = _first_existing(FAKE_SUMMARY_CANDIDATES)
    if path is None:
        if REQUIRE_FAKE_SOURCE_ERRORS:
            raise FileNotFoundError(
                "The fake-source recovery summary was not found. Run "
                "fake_source_proper_motion.ipynb first, then place "
                "ID639188_fake_source_recovery_summary.csv in one of the "
                "FAKE_SUMMARY_CANDIDATES paths."
            )
        return pd.DataFrame()

    fake = pd.read_csv(path)
    if "mode" in fake.columns:
        fake = fake[fake["mode"].astype(str).str.lower() == "moving"].copy()

    print("Using fake-source uncertainties:", path)
    return fake.reset_index(drop=True)


def _fake_row_for_pair(
    fake_summary: pd.DataFrame,
    epoch1: str,
    epoch2: str,
) -> Optional[pd.Series]:
    if fake_summary.empty:
        return None

    match = (
        (fake_summary["epoch1"].astype(str) == epoch1)
        & (fake_summary["epoch2"].astype(str) == epoch2)
    )
    if match.sum() == 0:
        return None
    return fake_summary.loc[match].iloc[0]


def _positive_value(row, names, default=np.nan) -> float:
    if row is None:
        return float(default)

    for name in names:
        if name in row.index:
            try:
                value = float(row[name])
            except Exception:
                continue
            if np.isfinite(value) and value > 0:
                return value
    return float(default)


def _pair_uncertainties(
    image_row: pd.Series,
    fake_summary: pd.DataFrame,
) -> dict:
    """
    Return 1-sigma-like robust-MAD uncertainties from moving fake-source
    recovery.

    For the displacement plane, use the direct recovered-minus-input scatter
    in ΔRA and ΔDec. Do not use the stationary false-motion p95 as an error bar:
    that quantity is a conservative null-test threshold, not a 1-sigma error.
    """
    epoch1 = str(image_row["epoch1"])
    epoch2 = str(image_row["epoch2"])
    fake_row = _fake_row_for_pair(fake_summary, epoch1, epoch2)

    # Preferred direct uncertainties from the fake-source recovery summary.
    sig_dra = _positive_value(
        fake_row,
        ["err_corr_dra_mas_sigma_mad"],
    )
    sig_ddec = _positive_value(
        fake_row,
        ["err_corr_ddec_mas_sigma_mad"],
    )
    sig_mu_ra = _positive_value(
        fake_row,
        ["err_mu_ra_masyr_sigma_mad"],
    )
    sig_mu_dec = _positive_value(
        fake_row,
        ["err_mu_dec_masyr_sigma_mad"],
    )
    sig_mu_total = _positive_value(
        fake_row,
        [
            "err_mu_total_masyr_sigma_mad",
            "err_mu_vector_masyr_sigma_mad",
        ],
    )

    # Convert between displacement and proper-motion errors only when a direct
    # quantity is unavailable.
    dt = float(image_row["dt_yr"])
    if not np.isfinite(sig_dra) and np.isfinite(sig_mu_ra):
        sig_dra = sig_mu_ra * dt
    if not np.isfinite(sig_ddec) and np.isfinite(sig_mu_dec):
        sig_ddec = sig_mu_dec * dt
    if not np.isfinite(sig_mu_ra) and np.isfinite(sig_dra) and dt > 0:
        sig_mu_ra = sig_dra / dt
    if not np.isfinite(sig_mu_dec) and np.isfinite(sig_ddec) and dt > 0:
        sig_mu_dec = sig_ddec / dt

    # Propagated total proper-motion error as a final fallback.
    if not np.isfinite(sig_mu_total):
        mu_ra = float(image_row["mu_ra_masyr"])
        mu_dec = float(image_row["mu_dec_masyr"])
        mu_total = math.hypot(mu_ra, mu_dec)
        if (
            mu_total > 0
            and np.isfinite(sig_mu_ra)
            and np.isfinite(sig_mu_dec)
        ):
            sig_mu_total = math.sqrt(
                (mu_ra / mu_total * sig_mu_ra) ** 2
                + (mu_dec / mu_total * sig_mu_dec) ** 2
            )

    if REQUIRE_FAKE_SOURCE_ERRORS:
        required = [sig_dra, sig_ddec, sig_mu_ra, sig_mu_dec, sig_mu_total]
        if not np.all(np.isfinite(required)):
            raise ValueError(
                f"Incomplete fake-source errors for {epoch1} -> {epoch2}. "
                "Check the moving-mode rows and column names in the recovery "
                "summary CSV."
            )

    return {
        "sig_dra_mas": sig_dra,
        "sig_ddec_mas": sig_ddec,
        "sig_mu_ra_masyr": sig_mu_ra,
        "sig_mu_dec_masyr": sig_mu_dec,
        "sig_mu_total_masyr": sig_mu_total,
    }


def _combined_main_result(
    image_summary: pd.DataFrame,
    fake_summary: pd.DataFrame,
) -> dict:
    """
    Prefer the Monte-Carlo combined result from fake_source_proper_motion.ipynb.
    If it is unavailable, calculate a component-wise inverse-variance estimate
    from the three main pairs.
    """
    combined_path = _first_existing(COMBINED_SUMMARY_CANDIDATES)
    if combined_path is not None:
        combined = pd.read_csv(combined_path)

        def get_quantity(name):
            row = combined[combined["quantity"].astype(str) == name]
            if len(row) != 1:
                return None
            row = row.iloc[0]
            med = float(row["median"])
            p16 = float(row["p16"])
            p84 = float(row["p84"])
            return med, med - p16, p84 - med

        total = get_quantity("mu_total_masyr")
        ra = get_quantity("mu_ra_masyr")
        dec = get_quantity("mu_dec_masyr")
        if total is not None and ra is not None and dec is not None:
            return {
                "mu_ra": ra[0],
                "mu_dec": dec[0],
                "mu_total": total[0],
                "mu_total_lo": total[1],
                "mu_total_hi": total[2],
                "source": str(combined_path),
            }

    rows = [_get_pair_row(image_summary, *pair) for pair in MAIN_PAIRS]
    errors = [_pair_uncertainties(row, fake_summary) for row in rows]

    mu_ra = np.array([float(row["mu_ra_masyr"]) for row in rows])
    mu_dec = np.array([float(row["mu_dec_masyr"]) for row in rows])
    sig_ra = np.array([err["sig_mu_ra_masyr"] for err in errors])
    sig_dec = np.array([err["sig_mu_dec_masyr"] for err in errors])

    w_ra = 1.0 / sig_ra**2
    w_dec = 1.0 / sig_dec**2
    mean_ra = float(np.sum(w_ra * mu_ra) / np.sum(w_ra))
    mean_dec = float(np.sum(w_dec * mu_dec) / np.sum(w_dec))
    err_ra = float(np.sqrt(1.0 / np.sum(w_ra)))
    err_dec = float(np.sqrt(1.0 / np.sum(w_dec)))

    mean_total = math.hypot(mean_ra, mean_dec)
    err_total = math.sqrt(
        (mean_ra / mean_total * err_ra) ** 2
        + (mean_dec / mean_total * err_dec) ** 2
    )

    return {
        "mu_ra": mean_ra,
        "mu_dec": mean_dec,
        "mu_total": mean_total,
        "mu_total_lo": err_total,
        "mu_total_hi": err_total,
        "source": "component-wise inverse-variance fallback",
    }


# -----------------------------------------------------------------------------
# Image preparation and styling
# -----------------------------------------------------------------------------

def _shared_epoch_norm(image1: np.ndarray, image2: np.ndarray) -> ImageNormalize:
    """
    One common normalization for the two epochs in a row. This preserves a fair
    brightness comparison while the asinh stretch reveals both source and sky.
    """
    values = np.concatenate(
        [
            np.asarray(image1, dtype=float).ravel(),
            np.asarray(image2, dtype=float).ravel(),
        ]
    )
    values = values[np.isfinite(values)]

    if values.size < 20:
        return ImageNormalize(vmin=0.0, vmax=1.0, stretch=AsinhStretch())

    vmin, vmax = np.nanpercentile(values, [0.5, 99.7])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin = float(np.nanmin(values))
        vmax = float(np.nanmax(values))

    return ImageNormalize(
        vmin=vmin,
        vmax=vmax,
        stretch=AsinhStretch(a=0.08),
        clip=True,
    )


def _robust_symmetric_difference_limit(image: np.ndarray) -> float:
    values = np.asarray(image, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 20:
        return 1.0

    med = np.nanmedian(values)
    sigma = 1.4826 * np.nanmedian(np.abs(values - med))
    percentile_limit = np.nanpercentile(np.abs(values - med), 99.2)

    candidates = [
        4.5 * sigma if np.isfinite(sigma) and sigma > 0 else np.nan,
        percentile_limit,
    ]
    limit = np.nanmax(candidates)
    if not np.isfinite(limit) or limit <= 0:
        limit = np.nanmax(np.abs(values))
    return float(limit)


def _prepare_zoom_triplet(row: pd.Series) -> dict:
    name1 = str(row["epoch1"])
    name2 = str(row["epoch2"])

    data1, wcs1, _ = load_image(name1)
    data2, wcs2, _ = load_image(name2)

    # The first-epoch measured position is the reference point for the overlay.
    coord1, _, _, _ = centroid_source(data1, wcs1, TARGET)

    dra_corr = float(row["cand_corr_dra_mas"])
    ddec_corr = float(row["cand_corr_ddec_mas"])
    coord2 = skycoord_with_offsets(coord1, dra_corr, ddec_corr)

    ref_dra = float(row.get("ref_med_dra_mas", 0.0))
    ref_ddec = float(row.get("ref_med_ddec_mas", 0.0))

    full1 = make_cutout(data1, wcs1, TARGET, FULL_SIZE_ARCSEC)
    aligned2 = align_epoch2_to_epoch1_grid(
        data2,
        wcs2,
        full1,
        ref_dra,
        ref_ddec,
    )

    scale = (
        robust_flux_scale(full1.data, aligned2, full1.wcs, TARGET)
        if AUTO_FLUX_SCALE_FOR_DIFF
        else 1.0
    )
    aligned2_scaled = scale * aligned2
    difference = aligned2_scaled - full1.data

    cut1 = make_cutout(
        full1.data,
        full1.wcs,
        TARGET,
        PAPER_ZOOM_SIZE_ARCSEC,
    )
    cut2 = make_cutout(
        aligned2_scaled,
        full1.wcs,
        TARGET,
        PAPER_ZOOM_SIZE_ARCSEC,
    )
    cutd = make_cutout(
        difference,
        full1.wcs,
        TARGET,
        PAPER_ZOOM_SIZE_ARCSEC,
    )

    return {
        "row": row,
        "name1": name1,
        "name2": name2,
        "filter": filter_from_epoch_name(name1),
        "cut1": cut1,
        "cut2": cut2,
        "cutd": cutd,
        "coord1": coord1,
        "coord2": coord2,
    }


def _add_centroid_overlay(ax, wcs, coord1, coord2) -> None:
    x1, y1 = world_to_pixel(wcs, coord1)
    x2, y2 = world_to_pixel(wcs, coord2)

    dark_outline = [
        pe.Stroke(linewidth=3.0, foreground="black"),
        pe.Normal(),
    ]

    epoch1_marker = ax.plot(
        x1,
        y1,
        marker="+",
        markersize=9.5,
        markeredgewidth=1.8,
        color="white",
        linestyle="none",
        zorder=7,
    )[0]
    epoch1_marker.set_path_effects(dark_outline)

    epoch2_marker = ax.plot(
        x2,
        y2,
        marker="x",
        markersize=8.5,
        markeredgewidth=1.8,
        color="#35d0ff",
        linestyle="none",
        zorder=8,
    )[0]
    epoch2_marker.set_path_effects(dark_outline)

    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops={
            "arrowstyle": "-|>",
            "color": "#35d0ff",
            "lw": 1.5,
            "mutation_scale": 8,
            "shrinkA": 1.5,
            "shrinkB": 1.5,
        },
        zorder=6,
    )


def _format_cutout_axes(
    ax,
    cut,
    show_xlabels: bool,
    show_ylabels: bool,
) -> None:
    set_pixel_axes_arcsec(ax, cut, TARGET)

    ax.tick_params(
        axis="both",
        which="both",
        direction="in",
        top=True,
        right=True,
        length=3.0,
        width=0.8,
        pad=2,
    )

    if not show_xlabels:
        ax.set_xlabel("")
        ax.set_xticklabels([])
    if not show_ylabels:
        ax.set_ylabel("")
        ax.set_yticklabels([])


# -----------------------------------------------------------------------------
# Figure 1: clean three-row image comparison
# -----------------------------------------------------------------------------

def make_main_image_figure() -> tuple[Path, Path]:
    image_summary = pd.read_csv(find_summary_csv())
    fake_summary = _load_fake_moving_summary()

    rows = [_get_pair_row(image_summary, *pair) for pair in MAIN_PAIRS]
    prepared = [_prepare_zoom_triplet(row) for row in rows]
    uncertainties = [
        _pair_uncertainties(row, fake_summary)
        for row in rows
    ]

    style = {
        "font.size": 8.5,
        "axes.titlesize": 9.3,
        "axes.labelsize": 8.2,
        "xtick.labelsize": 7.2,
        "ytick.labelsize": 7.2,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }

    with plt.rc_context(style):
        fig = plt.figure(figsize=(7.25, 5.85), facecolor="white")
        gs = GridSpec(
            nrows=3,
            ncols=4,
            figure=fig,
            width_ratios=[1.08, 1.0, 1.0, 1.0],
            left=0.025,
            right=0.995,
            bottom=0.095,
            top=0.93,
            wspace=0.045,
            hspace=0.085,
        )

        column_titles = [
            "PRIMER 2023",
            "COSMOS-3D 2025 aligned",
            r"2025 $-$ 2023",
        ]

        panel_index = 0

        for i, (item, err) in enumerate(zip(prepared, uncertainties)):
            row = item["row"]

            info_ax = fig.add_subplot(gs[i, 0])
            # Use axes-fraction coordinates explicitly. Without this transform,
            # Matplotlib interprets the text position in data coordinates. The
            # decorative line can then change the data limits, and bbox_inches="tight"
            # expands the canvas to include text placed far outside the intended cell.
            info_ax.set_xlim(0.0, 1.0)
            info_ax.set_ylim(0.0, 1.0)
            info_ax.axis("off")

            mu_total = float(row["mu_total_masyr"])
            dt = float(row["dt_yr"])
            survey1_key, year1 = item["name1"].split("_")[:2]
            survey2_key, year2 = item["name2"].split("_")[:2]
            survey1 = SURVEY_SHORT.get(survey1_key, survey1_key)
            survey2 = SURVEY_SHORT.get(survey2_key, survey2_key)

            # Separate text objects give more reliable vertical alignment than one
            # multiline math-text block.
            info_ax.text(
                0.93, 0.67, item["filter"],
                transform=info_ax.transAxes,
                ha="right", va="center",
                fontsize=9.0, fontweight="bold",
            )
            info_ax.text(
                0.93, 0.52,
                f"{survey1} ({year1})\n→ {survey2} ({year2})",
                transform=info_ax.transAxes,
                ha="right", va="center",
                linespacing=1.18, fontsize=8.0,
            )
            info_ax.text(
                0.93, 0.32,
                rf"$\Delta t={dt:.2f}$ yr" "\n"
                rf"$|\mu|={mu_total:.1f}\pm{err['sig_mu_total_masyr']:.1f}$ "
                r"mas yr$^{-1}$",
                transform=info_ax.transAxes,
                ha="right", va="center",
                linespacing=1.18, fontsize=8.0,
            )
            info_ax.plot(
                [0.985, 0.985], [0.10, 0.90],
                transform=info_ax.transAxes,
                color="0.72", lw=0.8,
                clip_on=False,
            )

            cuts = [item["cut1"], item["cut2"], item["cutd"]]
            epoch_norm = _shared_epoch_norm(
                item["cut1"].data,
                item["cut2"].data,
            )
            diff_limit = _robust_symmetric_difference_limit(
                item["cutd"].data
            )

            for j, cut in enumerate(cuts):
                ax = fig.add_subplot(gs[i, j + 1])

                if j < 2:
                    ax.imshow(
                        cut.data,
                        origin="lower",
                        cmap=SOURCE_CMAP,
                        norm=epoch_norm,
                        interpolation="none",
                        rasterized=True,
                    )
                else:
                    ax.imshow(
                        cut.data,
                        origin="lower",
                        cmap=DIFFERENCE_CMAP,
                        vmin=-diff_limit,
                        vmax=diff_limit,
                        interpolation="none",
                        rasterized=True,
                    )

                _add_centroid_overlay(
                    ax,
                    cut.wcs,
                    item["coord1"],
                    item["coord2"],
                )
                _format_cutout_axes(
                    ax,
                    cut,
                    show_xlabels=(i == len(prepared) - 1),
                    show_ylabels=(j == 0),
                )

                if i == 0:
                    ax.set_title(column_titles[j], pad=4)

                letter = string.ascii_lowercase[panel_index]
                ax.text(
                    0.035,
                    0.955,
                    f"({letter})",
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=8.2,
                    fontweight="bold",
                    color="black",
                    bbox={
                        "boxstyle": "round,pad=0.15",
                        "facecolor": "white",
                        "edgecolor": "none",
                        "alpha": 0.82,
                    },
                    zorder=10,
                )
                panel_index += 1

        fig.text(
            0.62,
            0.022,
            r"$+$: epoch-1 centroid; $\times$: epoch-2 centroid; "
            r"arrow: corrected displacement. "
            r"Each epoch pair shares one display scale.",
            ha="center",
            va="bottom",
            fontsize=7.1,
        )

        pdf_path = PAPER_OUTDIR / f"ID{OBJID}_proper_motion_main_image.pdf"
        png_path = PAPER_OUTDIR / f"ID{OBJID}_proper_motion_main_image.png"

        fig.savefig(pdf_path, facecolor="white")
        fig.savefig(png_path, dpi=400, facecolor="white")
        plt.show()
        plt.close(fig)

    print("Saved:", pdf_path)
    print("Saved:", png_path)
    return pdf_path, png_path


# -----------------------------------------------------------------------------
# Figure 2: quantitative astrometric summary with error bars
# -----------------------------------------------------------------------------

def make_quantitative_pm_figure() -> tuple[Path, Path]:
    image_summary = pd.read_csv(find_summary_csv())
    fake_summary = _load_fake_moving_summary()

    rows = [_get_pair_row(image_summary, *pair) for pair in QUANTITATIVE_PAIRS]
    errors = [_pair_uncertainties(row, fake_summary) for row in rows]
    combined = _combined_main_result(image_summary, fake_summary)

    markers = ["o", "s", "D", "^"]
    point_colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]

    style = {
        "font.size": 8.5,
        "axes.titlesize": 9.5,
        "axes.labelsize": 9.0,
        "xtick.labelsize": 7.4,
        "ytick.labelsize": 8.0,
        "legend.fontsize": 7.2,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }

    with plt.rc_context(style):
        fig, axes = plt.subplots(
            1,
            2,
            figsize=(7.25, 3.45),
            gridspec_kw={
                "left": 0.085,
                "right": 0.985,
                "bottom": 0.18,
                "top": 0.88,
                "wspace": 0.30,
            },
        )

        # -------------------------------------------------------------
        # (a) Corrected displacement plane
        # -------------------------------------------------------------
        ax = axes[0]

        all_x = [0.0]
        all_y = [0.0]

        for i, (row, err, pair) in enumerate(
            zip(rows, errors, QUANTITATIVE_PAIRS)
        ):
            x = float(row["cand_corr_dra_mas"])
            y = float(row["cand_corr_ddec_mas"])
            xerr = err["sig_dra_mas"]
            yerr = err["sig_ddec_mas"]

            label = PAIR_DISPLAY[pair]

            ax.errorbar(
                x,
                y,
                xerr=xerr,
                yerr=yerr,
                fmt=markers[i],
                markersize=6.3,
                markerfacecolor=point_colors[i],
                markeredgecolor="white",
                markeredgewidth=0.8,
                ecolor=point_colors[i],
                elinewidth=1.2,
                capsize=2.8,
                capthick=1.1,
                label=label,
                zorder=4,
            )
            all_x.extend([x - xerr, x + xerr])
            all_y.extend([y - yerr, y + yerr])

        ax.axhline(0.0, color="0.55", lw=0.8, zorder=1)
        ax.axvline(0.0, color="0.55", lw=0.8, zorder=1)
        ax.grid(alpha=0.18, lw=0.6)

        x_min = min(all_x)
        y_min = min(all_y)
        x_span = max(all_x) - x_min
        y_span = max(all_y) - y_min
        ax.set_xlim(x_min - 0.08 * max(x_span, 1.0), 10.0)
        ax.set_ylim(y_min - 0.10 * max(y_span, 1.0), 10.0)
        ax.set_aspect("equal", adjustable="box")

        ax.set_xlabel(r"Corrected $\Delta\alpha^\ast$ [mas]")
        ax.set_ylabel(r"Corrected $\Delta\delta$ [mas]")
        ax.set_title("(a) Corrected displacement")
        ax.legend(
            loc="upper right",
            frameon=True,
            facecolor="white",
            edgecolor="0.8",
            framealpha=0.95,
            handletextpad=0.5,
            borderaxespad=0.4,
        )

        # -------------------------------------------------------------
        # (b) Pair-by-pair |mu| and combined main-pair result
        # -------------------------------------------------------------
        ax = axes[1]

        x = np.arange(len(rows), dtype=float)
        mu = np.array([float(row["mu_total_masyr"]) for row in rows])
        mu_err = np.array([err["sig_mu_total_masyr"] for err in errors])

        for i in range(len(rows)):
            ax.errorbar(
                x[i],
                mu[i],
                yerr=mu_err[i],
                fmt=markers[i],
                markersize=6.5,
                markerfacecolor=point_colors[i],
                markeredgecolor="white",
                markeredgewidth=0.8,
                ecolor=point_colors[i],
                elinewidth=1.2,
                capsize=3.0,
                capthick=1.1,
                zorder=4,
            )

        combined_mu = float(combined["mu_total"])
        combined_lo = float(combined["mu_total_lo"])
        combined_hi = float(combined["mu_total_hi"])

        ax.axhspan(
            combined_mu - combined_lo,
            combined_mu + combined_hi,
            color="0.75",
            alpha=0.34,
            lw=0,
            label="Combined 68% interval",
            zorder=1,
        )
        ax.axhline(
            combined_mu,
            color="0.25",
            lw=1.1,
            linestyle="--",
            zorder=2,
        )

        labels = [
            "F115W\nPRIMER–C3D",
            "F200W\nPRIMER–C3D",
            "F356W\nPRIMER–C3D",
            "F115W\nCW–C3D",
        ]
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_xlim(-0.55, len(rows) - 0.45)

        upper = max(
            np.nanmax(mu + mu_err),
            combined_mu + combined_hi,
        )
        lower = min(
            np.nanmin(mu - mu_err),
            combined_mu - combined_lo,
        )
        margin = 0.16 * max(upper - lower, 1.0)
        ax.set_ylim(lower - margin, upper + margin)

        ax.set_ylabel(r"$|\mu|$ [mas yr$^{-1}$]")
        ax.set_title("(b) Proper motion by epoch pair")
        ax.grid(axis="y", alpha=0.20, lw=0.6)
        ax.legend(loc="best", frameon=False)

        pdf_path = PAPER_OUTDIR / f"ID{OBJID}_proper_motion_quantitative.pdf"
        png_path = PAPER_OUTDIR / f"ID{OBJID}_proper_motion_quantitative.png"

        fig.savefig(pdf_path, facecolor="white")
        fig.savefig(png_path, dpi=400, facecolor="white")
        plt.show()
        plt.close(fig)

    print("Saved:", pdf_path)
    print("Saved:", png_path)
    print("Combined-result source:", combined["source"])
    return pdf_path, png_path


# -----------------------------------------------------------------------------
# Produce the two recommended manuscript figures
# -----------------------------------------------------------------------------

main_image_pdf, main_image_png = make_main_image_figure()
quantitative_pdf, quantitative_png = make_quantitative_pm_figure()


Using fake-source uncertainties: ID639188_fake_source_PM_test/ID639188_fake_source_recovery_summary.csv


Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729580.833 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.421398 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412384141.197 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.701718 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416390764.218 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729589.007 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_main_image.pdf
Saved: ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_main_image.png
Using fake-source uncertainties: ID639188_fake_source_PM_test/ID639188_fake_source_recovery_summary.csv


Saved: ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_quantitative.pdf
Saved: ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_quantitative.png
Combined-result source: ID639188_fake_source_PM_test/ID639188_fake_source_combined_pm_summary.csv


In [10]:
# RUN THIS CELL: generate, save, and display both manuscript figures
%matplotlib inline
from IPython.display import display, Image as IPyImage

print("Starting paper-ready proper-motion figures...")
print("Image-level summary:", find_summary_csv())
print("Output directory:", PAPER_OUTDIR.resolve())

try:
    main_image_pdf, main_image_png = make_main_image_figure()
    quantitative_pdf, quantitative_png = make_quantitative_pm_figure()
except Exception as exc:
    print("\nFIGURE GENERATION FAILED:", type(exc).__name__, str(exc))
    print("\nCheck that these inputs exist relative to the notebook directory:")
    print("  jwst_image_cutouts_for_pm/*.fits")
    print("  proper_motion_image_level/image_level_pm_summary.csv")
    print("  ID639188_fake_source_PM_test/ID639188_fake_source_recovery_summary.csv")
    print("  ID639188_fake_source_PM_test/ID639188_fake_source_combined_pm_summary.csv")
    raise

print("\nCreated:")
print(" ", main_image_pdf)
print(" ", main_image_png)
print(" ", quantitative_pdf)
print(" ", quantitative_png)

display(IPyImage(filename=str(main_image_png)))
display(IPyImage(filename=str(quantitative_png)))


Starting paper-ready proper-motion figures...
Image-level summary: proper_motion_image_level/image_level_pm_summary.csv
Output directory: /home/bahareh/Desktop/Important_Codes/ID639188_Fig3_style_PM/paper_ready
Using fake-source uncertainties: ID639188_fake_source_PM_test/ID639188_fake_source_recovery_summary.csv


Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729580.833 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.421398 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412384141.197 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.701718 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416390764.218 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -13.397752 from OBSGEO-[XYZ].
Set OBSGEO-H to 1412051141.843 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to   -12.725978 from OBSGEO-[XYZ].
Set OBSGEO-H to 1416729589.007 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Saved: ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_main_image.pdf
Saved: ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_main_image.png
Using fake-source uncertainties: ID639188_fake_source_PM_test/ID639188_fake_source_recovery_summary.csv


Saved: ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_quantitative.pdf
Saved: ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_quantitative.png
Combined-result source: ID639188_fake_source_PM_test/ID639188_fake_source_combined_pm_summary.csv

Created:
  ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_main_image.pdf
  ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_main_image.png
  ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_quantitative.pdf
  ID639188_Fig3_style_PM/paper_ready/ID639188_proper_motion_quantitative.png


## Corrected header baselines and regenerated Figure 4

Run the four added cells below **after the existing plotting functions have been defined**. They do not alter the Figure 4 plotting function, style, colors, markers, labels, layout, or output filename.

The added workflow:

1. reads `MJD-AVG` and `JWST_X/Y/Z` from the original association-level JWST `i2d` headers;
2. audits the actual epoch midpoints and recomputes each baseline using a 365.25-day Julian year;
3. applies the 160 pc annual-parallax correction already described in the manuscript;
4. rebuilds the fake-source errors and three-filter Monte Carlo combination from the raw trial CSV; and
5. calls the existing `make_quantitative_pm_figure()` function unchanged.

Expected primary-baseline check from the named MAST products:

- F115W: 1.926895586 yr
- F200W: 1.926743464 yr
- F356W: 1.926895586 yr

Required local inputs are the same `image_level_pm_summary.csv` and `ID639188_fake_source_trials.csv` produced by the earlier proper-motion notebooks. If you already downloaded the full original `i2d` products, set `LOCAL_ORIGINAL_I2D_DIR` in the first added code cell; otherwise only the small header ranges are read from MAST.

In [ ]:
"""
DROP-IN FINAL CELL FOR proper_motion_fig3.ipynb

Run this after the existing cells have defined make_quantitative_pm_figure().
It reads the original JWST i2d primary headers, replaces the stale hard-coded
calendar-date baselines with MJD-AVG baselines, applies the same 160 pc annual-
parallax correction described in the paper, rebuilds the fake-source and
combined summaries, and finally calls the existing Figure 4 function without
changing any plotting style, colors, markers, layout, labels, or output names.
"""

from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen
import math

import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.time import Time
import astropy.units as u
from IPython.display import display, Image as IPyImage


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

JULIAN_YEAR_DAYS = 365.25
PARALLAX_DISTANCE_PC = 160.0
N_COMBINED_SAMPLES_CORRECTED = 100_000
COMBINED_RNG_SEED_CORRECTED = 639188

# Define the target locally so this correction block does not depend on the
# original setup cell having been executed in the current kernel.
TARGET_RA_DEG = 150.0634341
TARGET_DEC_DEG = 2.3552216
TARGET = SkyCoord(
    ra=TARGET_RA_DEG * u.deg,
    dec=TARGET_DEC_DEG * u.deg,
    frame="icrs",
)

# The numerical correction is self-contained, but the final plotting call
# deliberately reuses the notebook's original helpers to preserve Figure 4's
# exact style. Fail early with a useful instruction after a kernel restart.
_missing_original_helpers = [
    name
    for name in ("filter_from_epoch_name", "make_quantitative_pm_figure")
    if name not in globals()
]
if _missing_original_helpers:
    raise RuntimeError(
        "Run the original proper_motion_fig3.ipynb cells first (or choose "
        "Kernel > Restart & Run All), then run this final correction cell. "
        "Missing original helper(s): " + ", ".join(_missing_original_helpers)
    )

# Optional: point this to a directory containing the original full i2d files.
# Leave as None to read only their primary headers from public MAST products.
LOCAL_ORIGINAL_I2D_DIR = globals().get("LOCAL_ORIGINAL_I2D_DIR", None)

BASELINE_PATCH_OUTDIR = Path("proper_motion_corrected_baseline")
BASELINE_PATCH_OUTDIR.mkdir(parents=True, exist_ok=True)

# These are the exact association-level products used to make the cutouts.
# MJD-AVG and JWST_X/Y/Z are retained in these original primary headers, but the
# old cutout-writing cell discarded most non-WCS metadata.
EPOCH_PRODUCTS = {
    "PRIMER_2023_F115W": "jw01837-o003_t003_nircam_clear-f115w_i2d.fits",
    "PRIMER_2023_F200W": "jw01837-o003_t003_nircam_clear-f200w_i2d.fits",
    "PRIMER_2023_F356W": "jw01837-o003_t003_nircam_clear-f356w_i2d.fits",
    "COSMOSWeb_2023_F115W": "jw01727-o051_t051_nircam_clear-f115w_i2d.fits",
    "COSMOS3D_2025_F115W": "jw05893-o014_t014_nircam_clear-f115w_i2d.fits",
    "COSMOS3D_2025_F200W": "jw05893-o014_t014_nircam_clear-f200w_i2d.fits",
    "COSMOS3D_2025_F356W": "jw05893-o014_t014_nircam_clear-f356w_i2d.fits",
}

ORIGINAL_IMAGE_SUMMARY_CANDIDATES = [
    Path("proper_motion_image_level/image_level_pm_summary.csv"),
    Path("image_level_pm_summary.csv"),
    Path("/mnt/data/image_level_pm_summary.csv"),
]

ORIGINAL_FAKE_TRIAL_CANDIDATES = [
    Path("ID639188_fake_source_PM_test/ID639188_fake_source_trials.csv"),
    Path("ID639188_fake_source_trials.csv"),
    Path("/mnt/data/ID639188_fake_source_trials.csv"),
]

CORRECTED_IMAGE_SUMMARY_PATH = (
    BASELINE_PATCH_OUTDIR / "image_level_pm_summary_corrected_baseline_parallax.csv"
)
CORRECTED_FAKE_TRIALS_PATH = (
    BASELINE_PATCH_OUTDIR / "ID639188_fake_source_trials_corrected_baseline_parallax.csv"
)
CORRECTED_FAKE_SUMMARY_PATH = (
    BASELINE_PATCH_OUTDIR / "ID639188_fake_source_recovery_summary_corrected_baseline_parallax.csv"
)
CORRECTED_COMBINED_SAMPLES_PATH = (
    BASELINE_PATCH_OUTDIR / "ID639188_fake_source_combined_pm_samples_corrected.csv"
)
CORRECTED_COMBINED_SUMMARY_PATH = (
    BASELINE_PATCH_OUTDIR / "ID639188_fake_source_combined_pm_summary_corrected.csv"
)
HEADER_AUDIT_PATH = BASELINE_PATCH_OUTDIR / "ID639188_original_i2d_header_epoch_audit.csv"
PAIR_AUDIT_PATH = BASELINE_PATCH_OUTDIR / "ID639188_corrected_pair_values.csv"


# -----------------------------------------------------------------------------
# Small helpers
# -----------------------------------------------------------------------------

def _first_existing_baseline_patch(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


def _mast_product_url(product_filename):
    uri = f"mast:JWST/product/{product_filename}"
    return "https://mast.stsci.edu/api/v0.1/Download/file?uri=" + quote(
        uri, safe=":/"
    )


def _read_original_primary_header(product_filename):
    """Read only the original i2d primary header, locally or by HTTP range."""
    local_path = None
    if LOCAL_ORIGINAL_I2D_DIR is not None:
        candidate = Path(LOCAL_ORIGINAL_I2D_DIR) / product_filename
        if candidate.exists():
            local_path = candidate

    if local_path is not None:
        with fits.open(local_path, memmap=False, lazy_load_hdus=True) as hdul:
            header = hdul[0].header.copy()
            if len(hdul) > 1:
                header.update(hdul[1].header)
            return header, f"local:{local_path}"

    remote_url = _mast_product_url(product_filename)
    try:
        # A JWST association i2d can be tens of GB. Request only the first
        # 256 KiB and stop at the primary-header END card; no image pixels are
        # downloaded and no fsspec/aiohttp dependency is required.
        request = Request(
            remote_url,
            headers={
                "Range": "bytes=0-262143",
                "User-Agent": "ID639188-proper-motion-header-audit/1.0",
            },
        )
        with urlopen(request, timeout=60) as response:
            first_bytes = response.read(262144)

        def parse_one_header(blob, start_byte):
            cards = [
                blob[offset : offset + 80]
                for offset in range(start_byte, len(blob), 80)
            ]
            end_card_index = next(
                index
                for index, card in enumerate(cards)
                if card[:8].decode("ascii", errors="ignore").strip() == "END"
            )
            header_bytes_used = int(
                math.ceil(((end_card_index + 1) * 80) / 2880.0) * 2880
            )
            header_text = blob[
                start_byte : start_byte + header_bytes_used
            ].decode("ascii", errors="strict")
            return fits.Header.fromstring(header_text, sep=""), header_bytes_used

        primary_header, primary_bytes = parse_one_header(first_bytes, 0)
        # JWST association i2d timing and observatory-position keywords are
        # normally in the SCI extension header, immediately after the empty
        # primary HDU. Merge it exactly as the old image loader merged headers.
        science_header, _ = parse_one_header(first_bytes, primary_bytes)
        primary_header.update(science_header)
        return primary_header, "MAST primary+SCI headers"
    except Exception as exc:
        raise RuntimeError(
            f"Could not read the primary header for {product_filename}. "
            "Set LOCAL_ORIGINAL_I2D_DIR to a folder containing the original "
            "i2d files, or check network access to MAST."
        ) from exc


def _header_midpoint_mjd(header):
    """Use MJD-AVG/EXPMID; allow explicit midpoint fallbacks with provenance."""
    for key in ("MJD-AVG", "EXPMID"):
        value = header.get(key)
        if value is not None and np.isfinite(float(value)):
            return float(value), key

    for start_key, end_key in (
        ("MJD-BEG", "MJD-END"),
        ("EXPSTART", "EXPEND"),
    ):
        start = header.get(start_key)
        end = header.get(end_key)
        if start is not None and end is not None:
            return 0.5 * (float(start) + float(end)), f"midpoint({start_key},{end_key})"

    date_beg = header.get("DATE-BEG")
    date_end = header.get("DATE-END")
    if date_beg is not None and date_end is not None:
        t1 = Time(date_beg, format="isot", scale="utc")
        t2 = Time(date_end, format="isot", scale="utc")
        return float(0.5 * (t1.mjd + t2.mjd)), "midpoint(DATE-BEG,DATE-END)"

    raise KeyError(
        "No MJD-AVG, EXPMID, MJD-BEG/MJD-END, EXPSTART/EXPEND, "
        "or DATE-BEG/DATE-END keywords were found."
    )


def _robust_sigma_baseline_patch(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    median = np.nanmedian(values)
    return float(1.4826 * np.nanmedian(np.abs(values - median)))


def _percentile_or_nan_baseline_patch(values, q):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(np.nanpercentile(values, q)) if values.size else np.nan


def _proper_motion_pa_vectorized(mu_ra, mu_dec):
    return (np.degrees(np.arctan2(mu_ra, mu_dec)) + 360.0) % 360.0


# -----------------------------------------------------------------------------
# 1. Audit original headers and derive the correct epochs
# -----------------------------------------------------------------------------

header_rows = []
for epoch_name, product_filename in EPOCH_PRODUCTS.items():
    header, access_source = _read_original_primary_header(product_filename)
    midpoint_mjd, midpoint_keyword = _header_midpoint_mjd(header)

    mjd_beg = float(header.get("MJD-BEG", header.get("EXPSTART", np.nan)))
    mjd_end = float(header.get("MJD-END", header.get("EXPEND", np.nan)))
    if np.isfinite(mjd_beg) and np.isfinite(mjd_end):
        if not (mjd_beg <= midpoint_mjd <= mjd_end):
            raise ValueError(
                f"Midpoint for {epoch_name} is outside the header exposure range."
            )

    xyz = [header.get(key) for key in ("JWST_X", "JWST_Y", "JWST_Z")]
    if any(value is None for value in xyz):
        raise KeyError(
            f"JWST_X/Y/Z are missing from {product_filename}; the parallax "
            "correction described in the paper cannot be reproduced."
        )

    header_rows.append(
        {
            "epoch": epoch_name,
            "product": product_filename,
            "midpoint_keyword": midpoint_keyword,
            "mjd_avg": midpoint_mjd,
            "midpoint_utc": Time(midpoint_mjd, format="mjd", scale="utc").isot,
            "mjd_beg": mjd_beg,
            "mjd_end": mjd_end,
            "jwst_x_km": float(xyz[0]),
            "jwst_y_km": float(xyz[1]),
            "jwst_z_km": float(xyz[2]),
            "header_access": access_source,
        }
    )

header_audit = pd.DataFrame(header_rows).sort_values("mjd_avg").reset_index(drop=True)
header_audit.to_csv(HEADER_AUDIT_PATH, index=False)
epoch_metadata = header_audit.set_index("epoch")

if set(EPOCH_PRODUCTS) != set(epoch_metadata.index):
    raise RuntimeError("The header audit did not return exactly one row per epoch.")

print("Original i2d header midpoint audit:")
print(
    header_audit[
        ["epoch", "product", "midpoint_keyword", "mjd_avg", "midpoint_utc"]
    ].to_string(index=False)
)


In [ ]:
# -----------------------------------------------------------------------------
# 2. Recompute baselines, parallax-corrected displacements, and proper motions
# -----------------------------------------------------------------------------

original_image_summary_path = _first_existing_baseline_patch(
    ORIGINAL_IMAGE_SUMMARY_CANDIDATES
)
if original_image_summary_path is None:
    raise FileNotFoundError(
        "Could not find the original image_level_pm_summary.csv. Run the image-"
        "level proper-motion notebook first."
    )

original_image_summary = pd.read_csv(original_image_summary_path)
required_image_columns = {
    "epoch1",
    "epoch2",
    "cand_corr_dra_mas",
    "cand_corr_ddec_mas",
}
missing_image_columns = required_image_columns - set(original_image_summary.columns)
if missing_image_columns:
    raise KeyError(f"Missing image-summary columns: {sorted(missing_image_columns)}")

corrected_image_summary = original_image_summary.copy()
corrected_image_summary["dt_yr_old"] = pd.to_numeric(
    corrected_image_summary.get("dt_yr", np.nan), errors="coerce"
)

target_ra_rad = float(TARGET.ra.radian)
target_dec_rad = float(TARGET.dec.radian)
au_km = float(u.au.to(u.km))
parallax_mas = 1000.0 / PARALLAX_DISTANCE_PC


def _parallax_factors_from_epoch(epoch_name):
    row = epoch_metadata.loc[epoch_name]
    x = float(row["jwst_x_km"]) / au_km
    y = float(row["jwst_y_km"]) / au_km
    z = float(row["jwst_z_km"]) / au_km
    p_ra = x * math.sin(target_ra_rad) - y * math.cos(target_ra_rad)
    p_dec = (
        x * math.cos(target_ra_rad) * math.sin(target_dec_rad)
        + y * math.sin(target_ra_rad) * math.sin(target_dec_rad)
        - z * math.cos(target_dec_rad)
    )
    return float(p_ra), float(p_dec)


for idx, row in corrected_image_summary.iterrows():
    epoch1 = str(row["epoch1"])
    epoch2 = str(row["epoch2"])
    if epoch1 not in epoch_metadata.index or epoch2 not in epoch_metadata.index:
        raise KeyError(f"No original-header metadata for {epoch1} -> {epoch2}")

    mjd1 = float(epoch_metadata.loc[epoch1, "mjd_avg"])
    mjd2 = float(epoch_metadata.loc[epoch2, "mjd_avg"])
    baseline_days = mjd2 - mjd1
    dt_yr = baseline_days / JULIAN_YEAR_DAYS
    if not np.isfinite(dt_yr) or dt_yr <= 0:
        raise ValueError(f"Non-positive header baseline for {epoch1} -> {epoch2}")

    p_ra1, p_dec1 = _parallax_factors_from_epoch(epoch1)
    p_ra2, p_dec2 = _parallax_factors_from_epoch(epoch2)
    parallax_dra = parallax_mas * (p_ra2 - p_ra1)
    parallax_ddec = parallax_mas * (p_dec2 - p_dec1)

    local_dra = float(row["cand_corr_dra_mas"])
    local_ddec = float(row["cand_corr_ddec_mas"])
    intrinsic_dra = local_dra - parallax_dra
    intrinsic_ddec = local_ddec - parallax_ddec

    corrected_image_summary.loc[idx, "mjd_avg_epoch1"] = mjd1
    corrected_image_summary.loc[idx, "mjd_avg_epoch2"] = mjd2
    corrected_image_summary.loc[idx, "baseline_days"] = baseline_days
    corrected_image_summary.loc[idx, "dt_yr"] = dt_yr
    corrected_image_summary.loc[idx, "parallax_distance_pc"] = PARALLAX_DISTANCE_PC
    corrected_image_summary.loc[idx, "parallax_dra_mas"] = parallax_dra
    corrected_image_summary.loc[idx, "parallax_ddec_mas"] = parallax_ddec
    corrected_image_summary.loc[idx, "cand_localframe_dra_mas"] = local_dra
    corrected_image_summary.loc[idx, "cand_localframe_ddec_mas"] = local_ddec
    corrected_image_summary.loc[idx, "cand_corr_dra_mas"] = intrinsic_dra
    corrected_image_summary.loc[idx, "cand_corr_ddec_mas"] = intrinsic_ddec
    corrected_image_summary.loc[idx, "cand_total_mas"] = math.hypot(
        intrinsic_dra, intrinsic_ddec
    )
    corrected_image_summary.loc[idx, "mu_ra_masyr"] = intrinsic_dra / dt_yr
    corrected_image_summary.loc[idx, "mu_dec_masyr"] = intrinsic_ddec / dt_yr
    corrected_image_summary.loc[idx, "mu_total_masyr"] = (
        math.hypot(intrinsic_dra, intrinsic_ddec) / dt_yr
    )

corrected_image_summary.to_csv(CORRECTED_IMAGE_SUMMARY_PATH, index=False)

pair_audit_columns = [
    "epoch1",
    "epoch2",
    "dt_yr_old",
    "dt_yr",
    "baseline_days",
    "parallax_dra_mas",
    "parallax_ddec_mas",
    "mu_ra_masyr",
    "mu_dec_masyr",
    "mu_total_masyr",
]
pair_audit = corrected_image_summary[pair_audit_columns].copy()
pair_audit.to_csv(PAIR_AUDIT_PATH, index=False)

print("\nCorrected pair values (MJD-AVG baseline; d = 160 pc parallax):")
print(pair_audit.to_string(index=False))


In [ ]:
# -----------------------------------------------------------------------------
# 3. Rebuild fake-source errors in mas/yr using the corrected pair baselines
# -----------------------------------------------------------------------------

original_fake_trials_path = _first_existing_baseline_patch(
    ORIGINAL_FAKE_TRIAL_CANDIDATES
)
if original_fake_trials_path is None:
    raise FileNotFoundError(
        "The raw fake-source trial CSV is required for an exact corrected Figure 4. "
        "Run fake_source_proper_motion.ipynb first; do not reuse only the old "
        "mas/yr summary because its errors also contain the stale baseline."
    )

corrected_trials = pd.read_csv(original_fake_trials_path)
required_trial_columns = {
    "epoch1",
    "epoch2",
    "mode",
    "input_corr_dra_mas",
    "input_corr_ddec_mas",
    "rec_corr_dra_mas",
    "rec_corr_ddec_mas",
}
missing_trial_columns = required_trial_columns - set(corrected_trials.columns)
if missing_trial_columns:
    raise KeyError(f"Missing fake-trial columns: {sorted(missing_trial_columns)}")

corrected_trials["dt_yr_old"] = pd.to_numeric(
    corrected_trials.get("dt_yr", np.nan), errors="coerce"
)
corrected_trials["input_localframe_dra_mas"] = corrected_trials[
    "input_corr_dra_mas"
]
corrected_trials["input_localframe_ddec_mas"] = corrected_trials[
    "input_corr_ddec_mas"
]
corrected_trials["rec_localframe_dra_mas"] = corrected_trials[
    "rec_corr_dra_mas"
]
corrected_trials["rec_localframe_ddec_mas"] = corrected_trials[
    "rec_corr_ddec_mas"
]

for _, pair_row in corrected_image_summary.iterrows():
    epoch1 = str(pair_row["epoch1"])
    epoch2 = str(pair_row["epoch2"])
    pair_mask = (
        corrected_trials["epoch1"].astype(str).eq(epoch1)
        & corrected_trials["epoch2"].astype(str).eq(epoch2)
    )
    if not pair_mask.any():
        raise ValueError(f"No fake trials found for {epoch1} -> {epoch2}")

    dt_yr = float(pair_row["dt_yr"])
    parallax_dra = float(pair_row["parallax_dra_mas"])
    parallax_ddec = float(pair_row["parallax_ddec_mas"])
    moving_mask = pair_mask & corrected_trials["mode"].astype(str).str.lower().eq(
        "moving"
    )

    # Only moving fakes emulate the target's apparent displacement. The
    # stationary injections remain a detector/registration null test at zero.
    corrected_trials.loc[moving_mask, "input_corr_dra_mas"] -= parallax_dra
    corrected_trials.loc[moving_mask, "input_corr_ddec_mas"] -= parallax_ddec
    corrected_trials.loc[moving_mask, "rec_corr_dra_mas"] -= parallax_dra
    corrected_trials.loc[moving_mask, "rec_corr_ddec_mas"] -= parallax_ddec

    corrected_trials.loc[pair_mask, "dt_yr"] = dt_yr

    input_dra = corrected_trials.loc[pair_mask, "input_corr_dra_mas"].to_numpy(float)
    input_ddec = corrected_trials.loc[pair_mask, "input_corr_ddec_mas"].to_numpy(float)
    rec_dra = corrected_trials.loc[pair_mask, "rec_corr_dra_mas"].to_numpy(float)
    rec_ddec = corrected_trials.loc[pair_mask, "rec_corr_ddec_mas"].to_numpy(float)

    err_dra = rec_dra - input_dra
    err_ddec = rec_ddec - input_ddec
    input_mu_ra = input_dra / dt_yr
    input_mu_dec = input_ddec / dt_yr
    rec_mu_ra = rec_dra / dt_yr
    rec_mu_dec = rec_ddec / dt_yr
    input_mu_total = np.hypot(input_mu_ra, input_mu_dec)
    rec_mu_total = np.hypot(rec_mu_ra, rec_mu_dec)

    corrected_trials.loc[pair_mask, "input_mu_ra_masyr"] = input_mu_ra
    corrected_trials.loc[pair_mask, "input_mu_dec_masyr"] = input_mu_dec
    corrected_trials.loc[pair_mask, "input_mu_total_masyr"] = input_mu_total
    corrected_trials.loc[pair_mask, "rec_mu_ra_masyr"] = rec_mu_ra
    corrected_trials.loc[pair_mask, "rec_mu_dec_masyr"] = rec_mu_dec
    corrected_trials.loc[pair_mask, "rec_mu_total_masyr"] = rec_mu_total
    corrected_trials.loc[pair_mask, "rec_pa_deg_east_of_north"] = (
        _proper_motion_pa_vectorized(rec_mu_ra, rec_mu_dec)
    )
    corrected_trials.loc[pair_mask, "err_corr_dra_mas"] = err_dra
    corrected_trials.loc[pair_mask, "err_corr_ddec_mas"] = err_ddec
    corrected_trials.loc[pair_mask, "err_corr_total_mas"] = np.hypot(
        err_dra, err_ddec
    )
    corrected_trials.loc[pair_mask, "err_mu_ra_masyr"] = err_dra / dt_yr
    corrected_trials.loc[pair_mask, "err_mu_dec_masyr"] = err_ddec / dt_yr
    corrected_trials.loc[pair_mask, "err_mu_vector_masyr"] = (
        np.hypot(err_dra, err_ddec) / dt_yr
    )
    corrected_trials.loc[pair_mask, "err_mu_total_masyr"] = (
        rec_mu_total - input_mu_total
    )

corrected_trials.to_csv(CORRECTED_FAKE_TRIALS_PATH, index=False)


def _build_corrected_recovery_summary(trials, image_summary):
    summary_rows = []
    statistic_columns = [
        "err_corr_dra_mas",
        "err_corr_ddec_mas",
        "err_corr_total_mas",
        "err_mu_ra_masyr",
        "err_mu_dec_masyr",
        "err_mu_vector_masyr",
        "err_mu_total_masyr",
    ]

    for (epoch1, epoch2, mode), group in trials.groupby(
        ["epoch1", "epoch2", "mode"], sort=True
    ):
        image_match = image_summary[
            image_summary["epoch1"].astype(str).eq(str(epoch1))
            & image_summary["epoch2"].astype(str).eq(str(epoch2))
        ]
        if len(image_match) != 1:
            raise ValueError(
                f"Expected one corrected image row for {epoch1} -> {epoch2}; "
                f"found {len(image_match)}."
            )
        image_row = image_match.iloc[0]
        result = {
            "epoch1": epoch1,
            "epoch2": epoch2,
            "filter": filter_from_epoch_name(str(epoch1)),
            "mode": mode,
            "n_fake": len(group),
            "dt_yr": float(image_row["dt_yr"]),
            "observed_corr_dra_mas": float(image_row["cand_corr_dra_mas"]),
            "observed_corr_ddec_mas": float(image_row["cand_corr_ddec_mas"]),
            "observed_mu_ra_masyr": float(image_row["mu_ra_masyr"]),
            "observed_mu_dec_masyr": float(image_row["mu_dec_masyr"]),
            "observed_mu_total_masyr": float(image_row["mu_total_masyr"]),
        }

        for column in statistic_columns:
            values = pd.to_numeric(group[column], errors="coerce").to_numpy(float)
            result[f"{column}_median"] = _percentile_or_nan_baseline_patch(
                values, 50
            )
            result[f"{column}_sigma_mad"] = _robust_sigma_baseline_patch(values)
            result[f"{column}_p16"] = _percentile_or_nan_baseline_patch(values, 16)
            result[f"{column}_p84"] = _percentile_or_nan_baseline_patch(values, 84)
            result[f"{column}_p2p5"] = _percentile_or_nan_baseline_patch(
                values, 2.5
            )
            result[f"{column}_p97p5"] = _percentile_or_nan_baseline_patch(
                values, 97.5
            )

        if str(mode).lower() == "stationary":
            false_mu = pd.to_numeric(
                group["rec_mu_total_masyr"], errors="coerce"
            ).to_numpy(float)
            p95 = _percentile_or_nan_baseline_patch(false_mu, 95)
            result["stationary_false_mu_p95_masyr"] = p95
            result["observed_over_stationary_p95"] = (
                float(image_row["mu_total_masyr"]) / p95
                if np.isfinite(p95) and p95 > 0
                else np.nan
            )

        summary_rows.append(result)

    return pd.DataFrame(summary_rows)


corrected_fake_summary = _build_corrected_recovery_summary(
    corrected_trials, corrected_image_summary
)
corrected_fake_summary.to_csv(CORRECTED_FAKE_SUMMARY_PATH, index=False)


In [ ]:
# -----------------------------------------------------------------------------
# 4. Rebuild the combined three-filter Monte Carlo result
# -----------------------------------------------------------------------------

primary_pairs_for_combination = [
    ("PRIMER_2023_F115W", "COSMOS3D_2025_F115W"),
    ("PRIMER_2023_F200W", "COSMOS3D_2025_F200W"),
    ("PRIMER_2023_F356W", "COSMOS3D_2025_F356W"),
]

combination_inputs = []
raw_weights = []
for epoch1, epoch2 in primary_pairs_for_combination:
    image_match = corrected_image_summary[
        corrected_image_summary["epoch1"].astype(str).eq(epoch1)
        & corrected_image_summary["epoch2"].astype(str).eq(epoch2)
    ]
    fake_match = corrected_trials[
        corrected_trials["epoch1"].astype(str).eq(epoch1)
        & corrected_trials["epoch2"].astype(str).eq(epoch2)
        & corrected_trials["mode"].astype(str).str.lower().eq("moving")
    ].reset_index(drop=True)

    if len(image_match) != 1 or len(fake_match) < 10:
        raise ValueError(
            f"Incomplete corrected combination inputs for {epoch1} -> {epoch2}."
        )

    image_row = image_match.iloc[0]
    # The manuscript's combined amplitude is the inverse-variance combination
    # of the three scalar |mu| measurements. Use the empirical total-amplitude
    # scatter from the moving fake recoveries for those weights. Center the
    # recovery errors below because these injections are being used to estimate
    # scatter, not to impose a centroid-bias correction on the measured motion.
    sigma_total = _robust_sigma_baseline_patch(
        fake_match["err_mu_total_masyr"]
    )
    if not np.isfinite(sigma_total) or sigma_total <= 0:
        raise ValueError(f"Invalid fake-source scatter for {epoch1} -> {epoch2}")

    raw_weights.append(1.0 / sigma_total**2)
    combination_inputs.append((image_row, fake_match))

pair_weights = np.asarray(raw_weights, dtype=float)
pair_weights /= pair_weights.sum()

rng = np.random.default_rng(COMBINED_RNG_SEED_CORRECTED)
combined_mu_ra = np.zeros(N_COMBINED_SAMPLES_CORRECTED, dtype=float)
combined_mu_dec = np.zeros(N_COMBINED_SAMPLES_CORRECTED, dtype=float)
combined_mu_total = np.zeros(N_COMBINED_SAMPLES_CORRECTED, dtype=float)

for weight, (image_row, fake_match) in zip(pair_weights, combination_inputs):
    indices = rng.integers(
        0, len(fake_match), size=N_COMBINED_SAMPLES_CORRECTED
    )
    all_error_ra = fake_match["err_mu_ra_masyr"].to_numpy(float)
    all_error_dec = fake_match["err_mu_dec_masyr"].to_numpy(float)
    all_error_total = fake_match["err_mu_total_masyr"].to_numpy(float)
    error_ra = all_error_ra[indices] - np.nanmedian(all_error_ra)
    error_dec = all_error_dec[indices] - np.nanmedian(all_error_dec)
    error_total = all_error_total[indices] - np.nanmedian(all_error_total)
    combined_mu_ra += weight * (float(image_row["mu_ra_masyr"]) - error_ra)
    combined_mu_dec += weight * (float(image_row["mu_dec_masyr"]) - error_dec)
    combined_mu_total += weight * (
        float(image_row["mu_total_masyr"]) - error_total
    )

combined_pa = _proper_motion_pa_vectorized(combined_mu_ra, combined_mu_dec)

corrected_combined_samples = pd.DataFrame(
    {
        "sample": np.arange(N_COMBINED_SAMPLES_CORRECTED),
        "mu_ra_masyr": combined_mu_ra,
        "mu_dec_masyr": combined_mu_dec,
        "mu_total_masyr": combined_mu_total,
        "pa_deg_east_of_north": combined_pa,
    }
)
corrected_combined_samples.to_csv(CORRECTED_COMBINED_SAMPLES_PATH, index=False)

combined_summary_rows = []
for column in [
    "mu_ra_masyr",
    "mu_dec_masyr",
    "mu_total_masyr",
    "pa_deg_east_of_north",
]:
    values = corrected_combined_samples[column].to_numpy(float)
    combined_summary_rows.append(
        {
            "quantity": column,
            "median": _percentile_or_nan_baseline_patch(values, 50),
            "p16": _percentile_or_nan_baseline_patch(values, 16),
            "p84": _percentile_or_nan_baseline_patch(values, 84),
            "p2p5": _percentile_or_nan_baseline_patch(values, 2.5),
            "p97p5": _percentile_or_nan_baseline_patch(values, 97.5),
            "sigma_mad": _robust_sigma_baseline_patch(values),
        }
    )

corrected_combined_summary = pd.DataFrame(combined_summary_rows)
corrected_combined_summary.to_csv(CORRECTED_COMBINED_SUMMARY_PATH, index=False)

print("\nCorrected combined main-pair result:")
print(corrected_combined_summary.to_string(index=False))


# -----------------------------------------------------------------------------
# 5. Regenerate Figure 4 using the original plotting function unchanged
# -----------------------------------------------------------------------------

# The existing helper functions read these candidate lists at call time. Point
# them at corrected tables; make_quantitative_pm_figure() itself is untouched.
SUMMARY_CSV_CANDIDATES = [CORRECTED_IMAGE_SUMMARY_PATH]
FAKE_SUMMARY_CANDIDATES = [CORRECTED_FAKE_SUMMARY_PATH]
COMBINED_SUMMARY_CANDIDATES = [CORRECTED_COMBINED_SUMMARY_PATH]
REQUIRE_FAKE_SOURCE_ERRORS = True

print("\nRegenerating Figure 4 with the original style and output filename...")
quantitative_pdf, quantitative_png = make_quantitative_pm_figure()

print("\nCorrected Figure 4 files:")
print(" ", quantitative_pdf)
print(" ", quantitative_png)
print("Header audit:", HEADER_AUDIT_PATH)
print("Pair audit:", PAIR_AUDIT_PATH)

display(IPyImage(filename=str(quantitative_png)))
